<div dir="rtl" style="text-align: right; line-height: 1.8;">

# Phase 2 — Website Knowledge Ingestion V2

## Step 1 — Website Discovery

هدف این مرحله، کشف تمام مقالات وب‌سایت بدون محدودیت نویسنده است.

در این مرحله فقط لینک مقالات شناسایی و بررسی می‌شوند و هنوز هیچ مقاله‌ای دانلود یا پردازش نمی‌شود.

### اهداف

* شناسایی روش دریافت فهرست تمام مقالات سایت
* پیمایش تمام صفحات `Blog`
* استخراج لینک‌های `/Blog/Detail/`
* حذف لینک‌های تکراری
* استخراج شناسه مقاله
* ثبت تعداد صفحات و مقالات کشف‌شده
* ایجاد `Baseline` برای `Website Dataset` فاز ۲

### نکته

تعداد مقالات فاز ۱ یا نتایج قبلی فقط به‌عنوان `Baseline` استفاده می‌شوند و تعداد مقالات در کد `Hard-code` نخواهد شد.

</div>


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی روش کشف تمام مقالات سایت

در فاز ۱، دریافت مقالات براساس شناسه نویسنده انجام می‌شد.

در فاز ۲ باید محدودیت نویسنده حذف شود و تمام مقالات `Blog` سایت کشف شوند.

هدف این مرحله، پیدا کردن `Endpoint` یا مکانیزمی است که فهرست کامل `Blog` را صفحه‌بندی می‌کند.

در این مرحله فقط `Discovery` بررسی می‌شود و هنوز هیچ مقاله‌ای دانلود نمی‌شود.

</div>


<div style="direction: rtl; white-space: normal; line-height: 1;">

</div>

In [1]:
import requests

URL = "https://www.fintelligence.ir/Blog/GetBlogSearchResult"

payload = {
    "PageSize": "12",
    "PageNo": 1,
    "SearchKeyword": "",
    "SelectedBlogCategoryIDs": [],
    "SelectedAgeRangeIDs": [],
    "SelectedBlogContentTypeIDs": [],
    "SelectedBlogTagIDs": [],
    "SelectedBlogPostSortTypeID": "1",
}

response = requests.post(
    URL,
    json=payload,
    timeout=30,
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print(response.text[:2000])

Status: 200
Content-Type: application/json; charset=utf-8
{
  "html": "    <li>\r\n<div class=\"blog-post-list-item1\">\r\n    <figure class=\"blog-post-list-item-image-area\">\r\n        <a href=\"/Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA\" title=\"\" target=\"_blank\" class=\"blog-post-list-item-image\">\r\n            <img src=\"https://cdn.fintelligence.ir/Uploaded/Blog/Image/8b473e21-984e-4e34-ae35-1b4b42c596aa.jpg\" alt=\"معرفی کتاب شعر &#171;پول ولی تا ابد نیست&#187;\" />\r\n            \r\n        </a>\r\n\r\n    </figure>\r\n    <div class=\"blog-post-list-item-details-area\">\r\n        <h2 class=\"blog-post-list-item-title\">\r\n            <a href=\"/Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA\" target=\"_blank\" title=

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## استخراج لینک‌های مقالات از صفحه اول

در این مرحله `HTML` برگشتی از `Endpoint` بررسی می‌شود و فقط لینک‌های مربوط به صفحات مقاله استخراج می‌شوند.

هنوز `Pagination` اجرا نمی‌شود و فقط صفحه اول برای بررسی صحت ساختار آزمایش خواهد شد.

</div>


In [2]:
from bs4 import BeautifulSoup

data = response.json()
html = data.get("html", "")

soup = BeautifulSoup(html, "html.parser")

links = []

for a in soup.find_all("a", href=True):
    href = a["href"]

    if "/Blog/Detail/" in href:
        links.append(href)

unique_links = sorted(set(links))

print("All article links found:", len(links))
print("Unique article links:", len(unique_links))

for link in unique_links:
    print(link)

All article links found: 36
Unique article links: 12
/Blog/Detail/18EBA5C7/%D8%A2%D9%85%D8%A7%D8%B1%DB%8C-%D8%AA%DA%A9%D8%A7%D9%86-%D8%AF%D9%87%D9%86%D8%AF%D9%87-%D8%A7%D8%B2-%D8%A2%D8%AE%D8%B1%DB%8C%D9%86-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D9%81%D9%82%D8%B1-%D9%88-%D9%86%D8%A7%D8%A7%D9%85%D9%86%DB%8C-%D8%BA%D8%B0%D8%A7%DB%8C%DB%8C-%D8%AF%D8%B1-%D8%A7%DB%8C%D8%B1%D8%A7%D9%86
/Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA
/Blog/Detail/596080EF/%D8%B3%D9%86%D8%AC%D8%B4-%D8%AA%D8%A7%D8%B2%D9%87-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D8%AF%D8%A7%D9%86%D8%B4-%D9%85%D8%A7%D9%84%DB%8C-%D8%AF%D8%B1-%D8%A7%D8%AA%D8%AD%D8%A7%D8%AF%DB%8C%D9%87-%D8%A7%D8%B1%D9%88%D9%BE%D8%A7
/Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%85%D9%87%D8%A7%D8%B1%D8%AA%D9%87%D8%A7%DB%8C-%D8%B1%D8%A7%D9%87%D8%A7%D9%8

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست Pagination

در این مرحله `Pagination` فقط روی سه صفحه اول بررسی می‌شود.

هدف این است که مشخص شود با تغییر `PageNo`، مقالات متفاوت دریافت می‌شوند و روش `Discovery` برای پیمایش کل سایت قابل استفاده است.

هنوز `Crawl` کامل سایت انجام نمی‌شود.

</div>


In [3]:
from bs4 import BeautifulSoup
import requests

URL = "https://www.fintelligence.ir/Blog/GetBlogSearchResult"

for page_no in range(1, 4):
    payload = {
        "PageSize": "12",
        "PageNo": page_no,
        "SearchKeyword": "",
        "SelectedBlogCategoryIDs": [],
        "SelectedAgeRangeIDs": [],
        "SelectedBlogContentTypeIDs": [],
        "SelectedBlogTagIDs": [],
        "SelectedBlogPostSortTypeID": "1",
    }

    response = requests.post(
        URL,
        json=payload,
        timeout=30,
    )

    response.raise_for_status()

    html = response.json().get("html", "")
    soup = BeautifulSoup(html, "html.parser")

    links = {
        a["href"]
        for a in soup.find_all("a", href=True)
        if "/Blog/Detail/" in a["href"]
    }

    print(f"Page {page_no}: {len(links)} unique articles")

    for link in sorted(links):
        print(" ", link)

    print("-" * 80)

Page 1: 12 unique articles
  /Blog/Detail/18EBA5C7/%D8%A2%D9%85%D8%A7%D8%B1%DB%8C-%D8%AA%DA%A9%D8%A7%D9%86-%D8%AF%D9%87%D9%86%D8%AF%D9%87-%D8%A7%D8%B2-%D8%A2%D8%AE%D8%B1%DB%8C%D9%86-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D9%81%D9%82%D8%B1-%D9%88-%D9%86%D8%A7%D8%A7%D9%85%D9%86%DB%8C-%D8%BA%D8%B0%D8%A7%DB%8C%DB%8C-%D8%AF%D8%B1-%D8%A7%DB%8C%D8%B1%D8%A7%D9%86
  /Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA
  /Blog/Detail/596080EF/%D8%B3%D9%86%D8%AC%D8%B4-%D8%AA%D8%A7%D8%B2%D9%87-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D8%AF%D8%A7%D9%86%D8%B4-%D9%85%D8%A7%D9%84%DB%8C-%D8%AF%D8%B1-%D8%A7%D8%AA%D8%AD%D8%A7%D8%AF%DB%8C%D9%87-%D8%A7%D8%B1%D9%88%D9%BE%D8%A7
  /Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%85%D9%87%D8%A7%D8%B1%D8%AA%D9%87%D8%A7%DB%8C-%D8%B1%D8%A7%D9%87%D8%A7%D9%86%D8%AF%D8%A7%D8%B

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## استخراج شناسه پایدار مقاله

در این مرحله `Article ID` از مسیر `URL` استخراج می‌شود.

هدف این است که شناسایی مقاله فقط به عنوان یا `Slug` وابسته نباشد و برای `Deduplication` و `Versioning` یک شناسه پایدار داشته باشیم.

</div>


In [4]:
def extract_article_id(url):
    parts = url.strip("/").split("/")

    if len(parts) >= 3 and parts[0] == "Blog" and parts[1] == "Detail":
        return parts[2]

    return None


sample_links = sorted(unique_links)

for link in sample_links:
    article_id = extract_article_id(link)
    print(article_id, "->", link)

18EBA5C7 -> /Blog/Detail/18EBA5C7/%D8%A2%D9%85%D8%A7%D8%B1%DB%8C-%D8%AA%DA%A9%D8%A7%D9%86-%D8%AF%D9%87%D9%86%D8%AF%D9%87-%D8%A7%D8%B2-%D8%A2%D8%AE%D8%B1%DB%8C%D9%86-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D9%81%D9%82%D8%B1-%D9%88-%D9%86%D8%A7%D8%A7%D9%85%D9%86%DB%8C-%D8%BA%D8%B0%D8%A7%DB%8C%DB%8C-%D8%AF%D8%B1-%D8%A7%DB%8C%D8%B1%D8%A7%D9%86
3E933D80 -> /Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA
596080EF -> /Blog/Detail/596080EF/%D8%B3%D9%86%D8%AC%D8%B4-%D8%AA%D8%A7%D8%B2%D9%87-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D8%AF%D8%A7%D9%86%D8%B4-%D9%85%D8%A7%D9%84%DB%8C-%D8%AF%D8%B1-%D8%A7%D8%AA%D8%AD%D8%A7%D8%AF%DB%8C%D9%87-%D8%A7%D8%B1%D9%88%D9%BE%D8%A7
5D28B5FC -> /Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%85%D9%87%D8%A7%D8%B1%D8%AA%D9%87%D8%A7%DB%8C-%D8%B1%D8%A7%D9%87%D8%A7%D9%86%D8%

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ساخت URL کامل و Canonical

در این مرحله لینک‌های نسبی مقاله به `URL` کامل تبدیل می‌شوند.

همچنین برای هر مقاله یک `Canonical URL` مستقل از `Slug` ساخته می‌شود.

هدف این است که در مراحل بعدی `Manifest`، `Deduplication` و `Versioning` به مسیر متنی مقاله وابسته نباشند.

</div>


In [5]:
from urllib.parse import urljoin

BASE_URL = "https://www.fintelligence.ir"

for link in sample_links:
    article_id = extract_article_id(link)
    absolute_url = urljoin(BASE_URL, link)

    canonical_url = f"{BASE_URL}/Blog/Detail/{article_id}"

    print("Article ID:", article_id)
    print("Absolute URL:", absolute_url)
    print("Canonical URL:", canonical_url)
    print("-" * 80)

Article ID: 18EBA5C7
Absolute URL: https://www.fintelligence.ir/Blog/Detail/18EBA5C7/%D8%A2%D9%85%D8%A7%D8%B1%DB%8C-%D8%AA%DA%A9%D8%A7%D9%86-%D8%AF%D9%87%D9%86%D8%AF%D9%87-%D8%A7%D8%B2-%D8%A2%D8%AE%D8%B1%DB%8C%D9%86-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D9%81%D9%82%D8%B1-%D9%88-%D9%86%D8%A7%D8%A7%D9%85%D9%86%DB%8C-%D8%BA%D8%B0%D8%A7%DB%8C%DB%8C-%D8%AF%D8%B1-%D8%A7%DB%8C%D8%B1%D8%A7%D9%86
Canonical URL: https://www.fintelligence.ir/Blog/Detail/18EBA5C7
--------------------------------------------------------------------------------
Article ID: 3E933D80
Absolute URL: https://www.fintelligence.ir/Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA
Canonical URL: https://www.fintelligence.ir/Blog/Detail/3E933D80
--------------------------------------------------------------------------------
Article ID: 596080EF
Absolute URL: https://www.fintelligence.ir/Blog/Detail/596080EF/%D

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی نقطه پایان Pagination

در این مرحله رفتار `Endpoint` در صفحات انتهایی بررسی می‌شود.

هدف این است که `Crawler` بدون `Hard-code` کردن تعداد صفحات بتواند پایان `Blog` را به‌صورت خودکار تشخیص دهد.

</div>


In [6]:
from bs4 import BeautifulSoup
import requests

URL = "https://www.fintelligence.ir/Blog/GetBlogSearchResult"

for page_no in range(40, 56):
    payload = {
        "PageSize": "12",
        "PageNo": page_no,
        "SearchKeyword": "",
        "SelectedBlogCategoryIDs": [],
        "SelectedAgeRangeIDs": [],
        "SelectedBlogContentTypeIDs": [],
        "SelectedBlogTagIDs": [],
        "SelectedBlogPostSortTypeID": "1",
    }

    response = requests.post(
        URL,
        json=payload,
        timeout=30,
    )

    response.raise_for_status()

    html = response.json().get("html", "")
    soup = BeautifulSoup(html, "html.parser")

    links = {
        a["href"]
        for a in soup.find_all("a", href=True)
        if "/Blog/Detail/" in a["href"]
    }

    print(
        f"Page {page_no}: "
        f"html_length={len(html)}, "
        f"unique_articles={len(links)}"
    )

Page 40: html_length=22355, unique_articles=12
Page 41: html_length=24776, unique_articles=12
Page 42: html_length=22942, unique_articles=12
Page 43: html_length=22230, unique_articles=12
Page 44: html_length=21407, unique_articles=12
Page 45: html_length=1832, unique_articles=1
Page 46: html_length=0, unique_articles=0
Page 47: html_length=0, unique_articles=0
Page 48: html_length=0, unique_articles=0
Page 49: html_length=0, unique_articles=0
Page 50: html_length=0, unique_articles=0
Page 51: html_length=0, unique_articles=0
Page 52: html_length=0, unique_articles=0
Page 53: html_length=0, unique_articles=0
Page 54: html_length=0, unique_articles=0
Page 55: html_length=0, unique_articles=0


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه بررسی Pagination

نتیجه آزمایش صفحات انتهایی نشان داد:

* صفحات `1` تا `44` هر کدام شامل ۱۲ مقاله هستند.
* صفحه `45` شامل ۱ مقاله است.
* از صفحه `46` به بعد هیچ مقاله‌ای برگردانده نمی‌شود.

بنابراین تعداد فعلی مقالات قابل کشف سایت برابر است با:

`44 × 12 + 1 = 529`

این نتیجه، تعداد `529` مقاله را به‌عنوان `Baseline` فعلی تأیید می‌کند.

با این حال این عدد در `Crawler` ثابت نخواهد شد و پایان صفحات باید به‌صورت خودکار تشخیص داده شود.

</div>


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ساخت تابع کامل Website Discovery

در این مرحله منطق کشف تمام مقالات به یک تابع مستقل تبدیل می‌شود.

`Crawler` باید:

* از صفحه اول شروع کند.
* صفحات را یکی‌یکی دریافت کند.
* لینک‌های مقاله را استخراج کند.
* در اولین صفحه بدون مقاله متوقف شود.
* لینک‌های تکراری را حذف کند.
* تعداد صفحات بررسی‌شده را گزارش دهد.
* تعداد مقالات یکتا را گزارش دهد.

تعداد صفحات و تعداد مقالات نباید در کد `Hard-code` شوند.

</div>


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 09 — ساخت تابع کامل `Website Discovery`

در این مرحله، منطق کشف تمام مقالات سایت به یک تابع مستقل تبدیل می‌شود.

هدف این است که فرآیند `Website Discovery` بدون وابستگی به تعداد مشخصی از صفحات یا مقالات اجرا شود.

تابع `Crawler` باید:

- از صفحه اول شروع کند.
- صفحات را به‌ترتیب بررسی کند.
- لینک‌های مربوط به مقالات را استخراج کند.
- لینک‌های تکراری را حذف کند.
- در اولین صفحه‌ای که هیچ مقاله‌ای وجود ندارد متوقف شود.
- تعداد مقالات کشف‌شده در هر صفحه را گزارش کند.
- در پایان، تعداد کل مقالات یکتا را نمایش دهد.

تعداد صفحات و تعداد مقالات در کد `Hard-code` نمی‌شود و پایان `Pagination` به‌صورت خودکار تشخیص داده خواهد شد.

</div>

In [7]:
# Code Cell 09 - Full website discovery

from bs4 import BeautifulSoup
import requests


BLOG_SEARCH_URL = "https://www.fintelligence.ir/Blog/GetBlogSearchResult"


def discover_all_article_links():
    all_links = set()
    page_no = 1

    while True:
        payload = {
            "PageSize": "12",
            "PageNo": page_no,
            "SearchKeyword": "",
            "SelectedBlogCategoryIDs": [],
            "SelectedAgeRangeIDs": [],
            "SelectedBlogContentTypeIDs": [],
            "SelectedBlogTagIDs": [],
            "SelectedBlogPostSortTypeID": "1",
        }

        response = requests.post(
            BLOG_SEARCH_URL,
            json=payload,
            timeout=30,
        )

        response.raise_for_status()

        html = response.json().get("html", "")
        soup = BeautifulSoup(html, "html.parser")

        page_links = {
            a["href"]
            for a in soup.find_all("a", href=True)
            if "/Blog/Detail/" in a["href"]
        }

        print(
            f"Page {page_no}: "
            f"{len(page_links)} articles"
        )

        if not page_links:
            break

        all_links.update(page_links)

        page_no += 1

    return sorted(all_links)


all_article_links = discover_all_article_links()

print("-" * 80)
print("Total unique articles:", len(all_article_links))

Page 1: 12 articles
Page 2: 12 articles
Page 3: 12 articles
Page 4: 12 articles
Page 5: 12 articles
Page 6: 12 articles
Page 7: 12 articles
Page 8: 12 articles
Page 9: 12 articles
Page 10: 12 articles
Page 11: 12 articles
Page 12: 12 articles
Page 13: 12 articles
Page 14: 12 articles
Page 15: 12 articles
Page 16: 12 articles
Page 17: 12 articles
Page 18: 12 articles
Page 19: 12 articles
Page 20: 12 articles
Page 21: 12 articles
Page 22: 12 articles
Page 23: 12 articles
Page 24: 12 articles
Page 25: 12 articles
Page 26: 12 articles
Page 27: 12 articles
Page 28: 12 articles
Page 29: 12 articles
Page 30: 12 articles
Page 31: 12 articles
Page 32: 12 articles
Page 33: 12 articles
Page 34: 12 articles
Page 35: 12 articles
Page 36: 12 articles
Page 37: 12 articles
Page 38: 12 articles
Page 39: 12 articles
Page 40: 12 articles
Page 41: 12 articles
Page 42: 12 articles
Page 43: 12 articles
Page 44: 12 articles
Page 45: 1 articles
Page 46: 0 articles
---------------------------------------------

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 10 — نتیجه `Website Discovery`

اجرای کامل تابع `discover_all_article_links()` نشان داد که فرآیند کشف مقالات سایت به‌درستی و بدون وابستگی به تعداد ثابت صفحات انجام می‌شود.

نتیجه اجرا:

- صفحات دارای مقاله: `45`
- اولین صفحه خالی: `46`
- تعداد کل مقالات یکتا: `529`

این نتیجه نشان می‌دهد که `Crawler` می‌تواند پایان `Pagination` را به‌صورت خودکار تشخیص دهد.

عدد `529` فقط نتیجه اجرای فعلی است و در منطق برنامه به‌صورت `Hard-code` استفاده نمی‌شود.

این خروجی به‌عنوان `Website Discovery Baseline` فاز ۲ ثبت می‌شود.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 11 — جداسازی فضای داده فاز ۲

برای جلوگیری از تداخل داده‌های فاز ۱ و فاز ۲، تمام داده‌های تولیدشده در فاز ۲ داخل مسیر مستقل `data/phase2/` نگهداری می‌شوند.

ساختار اولیه پیشنهادی:

`data/phase2/raw/`  
برای نگهداری نسخه خام و تغییرناپذیر منابع.

`data/phase2/interim/`  
برای خروجی اولیه استخراج متن و Metadata.

`data/phase2/processed/`  
برای داده‌های پاک‌سازی و استانداردشده.

`data/phase2/chunks/`  
برای Chunkهای آماده مراحل بعدی.

`data/phase2/manifests/`  
برای ثبت وضعیت، شناسه، نسخه و مشخصات منابع.

`data/phase2/evaluation/`  
برای Datasetها و نتایج ارزیابی.

همچنین گزارش‌های فاز ۲ به‌صورت مستقل در مسیر `reports/phase2/` نگهداری خواهند شد.

این جداسازی باعث می‌شود داده‌ها و نتایج فاز ۱ بدون تغییر باقی بمانند و تمام مراحل فاز ۲ نیز به‌صورت مستقل، قابل ردیابی و قابل بازسازی باشند.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 12 — ساخت ساختار پوشه‌های فاز ۲

در این مرحله ساختار اولیه داده‌ها و گزارش‌های فاز ۲ ایجاد می‌شود.

هدف این است که تمام خروجی‌های فاز ۲ از داده‌های فاز ۱ جدا باشند و مسیرهای مستقل برای داده خام، داده میانی، داده پردازش‌شده، Manifest و گزارش‌ها داشته باشیم.

در این مرحله فقط پوشه‌ها ایجاد می‌شوند و هنوز هیچ مقاله‌ای ذخیره یا پردازش نمی‌شود.

</div>

In [8]:
# Code Cell 10 - Create Phase 2 directories

from pathlib import Path


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "phase2":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


DATA_ROOT = PROJECT_ROOT / "data"
REPORTS_ROOT = PROJECT_ROOT / "reports"


phase2_directories = [
    DATA_ROOT / "phase2/raw/website",
    DATA_ROOT / "phase2/interim",
    DATA_ROOT / "phase2/processed",
    DATA_ROOT / "phase2/chunks",
    DATA_ROOT / "phase2/manifests",
    DATA_ROOT / "phase2/evaluation",
    REPORTS_ROOT / "phase2/ingestion",
    REPORTS_ROOT / "phase2/data_quality",
    REPORTS_ROOT / "phase2/benchmarks",
]


print("Project root:", PROJECT_ROOT)
print()

for directory in phase2_directories:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        f"Created or exists: {directory}"
    )


Project root: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance

Created or exists: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website
Created or exists: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/interim
Created or exists: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/processed
Created or exists: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/chunks
Created or exists: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/manifests
Created or exists: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/evaluation
Created or exists: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/reports/phase2/ingestion
Created or exists: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/reports/phase2/data_quality
Created or exists: /Users/macbookpro/AMIR_DATA/

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 13 — نتیجه ساخت ساختار فاز ۲

ساختار اولیه پوشه‌های فاز ۲ با موفقیت ایجاد شد.

مسیرهای داده و گزارش‌های فاز ۲ اکنون از فاز ۱ جدا هستند و شامل این بخش‌ها می‌شوند:

- `data/phase2/raw/website`
- `data/phase2/interim`
- `data/phase2/processed`
- `data/phase2/chunks`
- `data/phase2/manifests`
- `data/phase2/evaluation`
- `reports/phase2/ingestion`
- `reports/phase2/data_quality`
- `reports/phase2/benchmarks`

این جداسازی باعث می‌شود داده‌های فاز ۱ بدون تغییر باقی بمانند و تمام خروجی‌های فاز ۲ به‌صورت مستقل قابل ردیابی و بازسازی باشند.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 14 — ساخت `Website Manifest` اولیه

در این مرحله برای تمام مقالات کشف‌شده یک `Manifest` اولیه ساخته می‌شود.

هدف `Manifest` این است که وضعیت هر منبع در طول مراحل `Ingestion` قابل ردیابی باشد.

در این مرحله هنوز مقاله‌ای دانلود نمی‌شود و فقط اطلاعات حاصل از `Discovery` ثبت می‌شوند.

برای هر مقاله این اطلاعات اولیه نگهداری می‌شود:

- `source_id`
- `source_type`
- `article_id`
- `url`
- `canonical_url`
- `status`
- `version`

وضعیت اولیه تمام رکوردها:

`discovered`

خواهد بود.

</div>

In [9]:
# Code Cell 11 - Build initial website manifest

from urllib.parse import urljoin


BASE_URL = "https://www.fintelligence.ir"


website_manifest = []

for link in all_article_links:
    article_id = extract_article_id(link)

    absolute_url = urljoin(BASE_URL, link)
    canonical_url = f"{BASE_URL}/Blog/Detail/{article_id}"

    record = {
        "source_id": f"website:{article_id}",
        "source_type": "website",
        "article_id": article_id,
        "url": absolute_url,
        "canonical_url": canonical_url,
        "status": "discovered",
        "version": 1,
    }

    website_manifest.append(record)


print("Manifest records:", len(website_manifest))
print()

for record in website_manifest[:3]:
    print(record)
    print("-" * 80)

Manifest records: 529

{'source_id': 'website:008E61CC', 'source_type': 'website', 'article_id': '008E61CC', 'url': 'https://www.fintelligence.ir/Blog/Detail/008E61CC/%D8%B9%D9%84%D9%81-%D8%AF%D8%A7%D8%AF%D9%85-%D8%B4%DB%8C%D8%B1-%DA%AF%D8%B1%D9%81%D8%AA%D9%85', 'canonical_url': 'https://www.fintelligence.ir/Blog/Detail/008E61CC', 'status': 'discovered', 'version': 1}
--------------------------------------------------------------------------------
{'source_id': 'website:00D061A8', 'source_type': 'website', 'article_id': '00D061A8', 'url': 'https://www.fintelligence.ir/Blog/Detail/00D061A8/%D9%88%D9%82%D8%A7%DB%8C%D8%B9-%D9%86%DA%AF%D8%A7%D8%B1%DB%8C-%D9%87%D8%A7%DB%8C-%D8%B3%D9%86%D8%AC%D8%B4-%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-10', 'canonical_url': 'https://www.fintelligence.ir/Blog/Detail/00D061A8', 'status': 'discovered', 'version': 1}
--------------------------------------------------------------------------------
{'source_id': 'website:00D9A87F', 'source_type': 'webs

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 15 — اعتبارسنجی `Website Manifest`

قبل از ذخیره `Manifest`، ساختار آن اعتبارسنجی می‌شود.

هدف این مرحله بررسی این موارد است:

- تعداد رکوردها با تعداد مقالات کشف‌شده برابر باشد.
- هیچ `article_id` خالی وجود نداشته باشد.
- هیچ `source_id` تکراری وجود نداشته باشد.
- هیچ `canonical_url` تکراری وجود نداشته باشد.
- تمام رکوردها در وضعیت اولیه `discovered` باشند.

در این مرحله هنوز `Manifest` روی دیسک ذخیره نمی‌شود.

</div>

In [10]:
# Code Cell 12 - Validate initial website manifest

article_ids = [record["article_id"] for record in website_manifest]
source_ids = [record["source_id"] for record in website_manifest]
canonical_urls = [record["canonical_url"] for record in website_manifest]
statuses = [record["status"] for record in website_manifest]


print("Total records:", len(website_manifest))
print("Missing article IDs:", sum(article_id is None for article_id in article_ids))
print("Unique article IDs:", len(set(article_ids)))
print("Unique source IDs:", len(set(source_ids)))
print("Unique canonical URLs:", len(set(canonical_urls)))
print("Discovered status count:", statuses.count("discovered"))

Total records: 529
Missing article IDs: 0
Unique article IDs: 529
Unique source IDs: 529
Unique canonical URLs: 529
Discovered status count: 529


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 16 — تفکیک `Author` و `Knowledge Ownership`

در فاز ۲ باید میان مقاله‌هایی که مستقیماً توسط دکتر کمیل رودی نوشته شده‌اند و مقاله‌هایی که زیرنظر ایشان تولید شده‌اند تفاوت قائل شویم.

برای این منظور سه فیلد به `Manifest` اضافه می‌شود:

- `author`
- `knowledge_owner`
- `content_origin`

مقدار `knowledge_owner` برای مجموعه مقالات این سایت برابر با دکتر کمیل رودی در نظر گرفته می‌شود.

مقدار `content_origin` یکی از این حالت‌ها خواهد بود:

- `primary_author`
- `supervised`
- `unknown`

در این مرحله هنوز نویسنده تمام ۵۲۹ مقاله از صفحه مقاله استخراج نمی‌شود.

فعلاً فقط مقاله‌های شناخته‌شده دکتر کمیل رودی با استفاده از فهرست فاز ۱ علامت‌گذاری می‌شوند و بقیه موقتاً در گروه `supervised` قرار می‌گیرند.

</div>

In [11]:
# Code Cell 13 - Locate Phase 1 Komeil Roudi data

from pathlib import Path


search_targets = [
    Path("data/raw/webpages/komeil_roudi_links.txt"),
    Path("data/raw/webpages/komeil_roudi"),
]

for target in search_targets:
    print(f"{target} -> exists: {target.exists()}")

data/raw/webpages/komeil_roudi_links.txt -> exists: False
data/raw/webpages/komeil_roudi -> exists: False


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 17 — بررسی داده مرجع مقالات دکتر کمیل رودی

بررسی مسیرهای مورد انتظار فاز ۱ نشان داد که فایل فهرست مقالات و پوشه مقالات دکتر کمیل رودی در وضعیت فعلی Repository وجود ندارند.

مسیرهای بررسی‌شده:

- `data/raw/webpages/komeil_roudi_links.txt`
- `data/raw/webpages/komeil_roudi/`

بنابراین برای تشخیص مقاله‌های `primary_author` نباید به فایل محلی‌ای که وجود ندارد وابسته شویم.

در مرحله بعد، فهرست مقالات دکتر کمیل رودی مستقیماً از همان روش استخراج معتبر فاز ۱ بازیابی می‌شود و سپس شناسه‌های آن با ۵۲۹ مقاله کشف‌شده تطبیق داده خواهند شد.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## سلول Markdown 18 — بازیابی مقالات مستقیم دکتر کمیل رودی

برای تفکیک مقاله‌هایی که مستقیماً توسط دکتر کمیل رودی نوشته شده‌اند، از `Author Endpoint` سایت استفاده می‌شود.

در فاز ۱ شناسه نویسنده دکتر کمیل رودی برابر با `4` بوده است.

در این مرحله فقط صفحه اول این منبع دریافت می‌شود تا ساختار پاسخ و امکان استخراج لینک مقالات مجدداً اعتبارسنجی شود.

هنوز کل مقالات نویسنده دریافت یا در `Manifest` طبقه‌بندی نمی‌شوند.

</div>

In [12]:
# Code Cell 13 - Test Komeil Roudi author endpoint

from bs4 import BeautifulSoup
import requests


BASE_URL = "https://www.fintelligence.ir"
AUTHOR_ID = 4

author_url = (
    f"{BASE_URL}/Author/GetAuthorBlogByPage/1"
    f"?auid={AUTHOR_ID}"
)

response = requests.get(
    author_url,
    timeout=30,
)

response.raise_for_status()

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("Response length:", len(response.text))
print()
print(response.text[:1000])

Status: 200
Content-Type: text/html; charset=utf-8
Response length: 17821


    <li class="li-blog-post-list-item1">
<div class="blog-post-list-item1">
    <figure class="blog-post-list-item-image-area">
        <a href="/Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%85%D9%87%D8%A7%D8%B1%D8%AA%D9%87%D8%A7%DB%8C-%D8%B1%D8%A7%D9%87%D8%A7%D9%86%D8%AF%D8%A7%D8%B2%DB%8C-%DA%A9%D8%B3%D8%A8%D9%88%DA%A9%D8%A7%D8%B1" title="" target="_blank" class="blog-post-list-item-image">
            <img src="https://cdn.fintelligence.ir/Uploaded/Blog/Image/90b08ea6-068a-432c-babe-0ce14574ff6c.jpg" alt="سواد مالی برای خانم‌ها؛ مهارت‌های راه‌اندازی کسب‌وکار" />
            
        </a>

    </figure>
    <div class="blog-post-list-item-details-area">
        <h2 class="blog-post-list-item-title">
            <a href="/Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## استخراج لینک مقالات دکتر کمیل رودی

در این مرحله لینک‌های مقاله از پاسخ `Author Endpoint` استخراج می‌شوند.

هدف این تست:

- شناسایی لینک‌های `/Blog/Detail/`
- حذف لینک‌های تکراری داخل همان صفحه
- استخراج تعداد واقعی مقاله‌های صفحه اول
- آماده‌سازی برای پیمایش کامل صفحات نویسنده

هنوز کل صفحات نویسنده پیمایش نمی‌شوند.

</div>

In [13]:
# Code Cell 14 - Extract article links from author page

soup = BeautifulSoup(response.text, "html.parser")

author_page_links = {
    a["href"]
    for a in soup.find_all("a", href=True)
    if "/Blog/Detail/" in a["href"]
}

print("Unique articles on author page 1:", len(author_page_links))
print()

for link in sorted(author_page_links):
    print(link)

Unique articles on author page 1: 8

/Blog/Detail/23C01D17/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%86%D8%AD%D9%88%DB%80-%D8%AF%D8%B3%D8%AA%D8%B1%D8%B3%DB%8C-%D9%85%D8%B4%D8%AA%D8%B1%DB%8C-%D8%A8%D9%87-%D9%85%D8%AD%D8%B5%D9%88%D9%84-%D9%85%D8%A7
/Blog/Detail/287D19E4/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D8%B3%D8%B1%D9%85%D8%A7%DB%8C%D9%87-%D8%A7%D9%86%D8%B3%D8%A7%D9%86%DB%8C
/Blog/Detail/5635AA40/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D8%B4%D8%B1%DA%A9%D8%A7%DB%8C-%DA%A9%D8%B3%D8%A8-%D9%88-%DA%A9%D8%A7%D8%B1
/Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%85%D9%87%D8%A7%D8%B1%D8%AA%D9%87%D8%A7%DB%8C-%D8%B1%D8%A7%D9%87%D8%A7%D9%86%D8%AF%D8%A7%D8%B2%DB%8C-%DA%A9%D8%B3%D8%A8%D9%8

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## دریافت کامل مقالات دکتر کمیل رودی

در این مرحله تمام صفحات `Author Endpoint` پیمایش می‌شوند و شناسه مقاله‌هایی که مستقیماً به دکتر کمیل رودی نسبت داده شده‌اند استخراج می‌شود.

پیمایش تا اولین صفحه بدون مقاله ادامه پیدا می‌کند و تعداد صفحات یا مقالات در کد `Hard-code` نمی‌شود.

خروجی این مرحله برای تفکیک مقالات `primary_author` از محتوای `supervised` در `Manifest` استفاده خواهد شد.

</div>

In [14]:
# Code Cell 15 - Discover all Komeil Roudi articles

def discover_author_article_links(author_id):
    all_links = set()
    page_no = 1

    while True:
        author_url = (
            f"{BASE_URL}/Author/GetAuthorBlogByPage/{page_no}"
            f"?auid={author_id}"
        )

        response = requests.get(
            author_url,
            timeout=30,
        )

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        page_links = {
            a["href"]
            for a in soup.find_all("a", href=True)
            if "/Blog/Detail/" in a["href"]
        }

        print(
            f"Page {page_no}: "
            f"{len(page_links)} articles"
        )

        if not page_links:
            break

        all_links.update(page_links)
        page_no += 1

    return sorted(all_links)


komeil_roudi_links = discover_author_article_links(AUTHOR_ID)

komeil_roudi_article_ids = {
    extract_article_id(link)
    for link in komeil_roudi_links
    if extract_article_id(link)
}

print("-" * 80)
print("Total Komeil Roudi articles:", len(komeil_roudi_links))
print("Unique article IDs:", len(komeil_roudi_article_ids))

Page 1: 8 articles
Page 2: 8 articles
Page 3: 8 articles
Page 4: 8 articles
Page 5: 8 articles
Page 6: 8 articles
Page 7: 8 articles
Page 8: 8 articles
Page 9: 8 articles
Page 10: 4 articles
Page 11: 0 articles
--------------------------------------------------------------------------------
Total Komeil Roudi articles: 76
Unique article IDs: 76


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## طبقه‌بندی مقالات بر اساس نقش محتوایی

در این مرحله، `Website Manifest` با استفاده از فهرست ۷۶ مقاله دکتر کمیل رودی غنی‌تر می‌شود.

برای مقاله‌هایی که شناسه آن‌ها در فهرست دکتر رودی وجود دارد:

- `author` برابر با دکتر کمیل رودی
- `knowledge_owner` برابر با دکتر کمیل رودی
- `content_origin` برابر با `primary_author`

برای سایر مقالات:

- `author` فعلاً `None`
- `knowledge_owner` برابر با دکتر کمیل رودی
- `content_origin` برابر با `supervised`

نویسنده واقعی سایر مقالات بعداً در مرحله `Article Extraction` از خود صفحه مقاله استخراج خواهد شد.

</div>

In [15]:
# Code Cell 16 - Classify manifest content origin

for record in website_manifest:
    article_id = record["article_id"]

    if article_id in komeil_roudi_article_ids:
        record["author"] = "Dr. Komeil Roudi"
        record["knowledge_owner"] = "Dr. Komeil Roudi"
        record["content_origin"] = "primary_author"
    else:
        record["author"] = None
        record["knowledge_owner"] = "Dr. Komeil Roudi"
        record["content_origin"] = "supervised"


primary_author_count = sum(
    record["content_origin"] == "primary_author"
    for record in website_manifest
)

supervised_count = sum(
    record["content_origin"] == "supervised"
    for record in website_manifest
)


print("Primary author articles:", primary_author_count)
print("Supervised articles:", supervised_count)
print("Total:", len(website_manifest))

Primary author articles: 76
Supervised articles: 453
Total: 529


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه طبقه‌بندی مالکیت دانشی مقالات

از مجموع `529` مقاله کشف‌شده:

- `76` مقاله در گروه `primary_author` قرار گرفتند.
- `453` مقاله در گروه `supervised` قرار گرفتند.

در گروه `primary_author`، نویسنده و مالک دانشی هر دو دکتر کمیل رودی ثبت شده‌اند.

در گروه `supervised`، نویسنده واقعی مقاله فعلاً مشخص نشده و بعداً از خود صفحه مقاله استخراج می‌شود، اما `knowledge_owner` برابر با دکتر کمیل رودی باقی می‌ماند.

این تفکیک در مراحل بعدی برای `Retrieval`، `Evaluation`، `Metadata Filtering` و تحلیل کیفیت منابع قابل استفاده خواهد بود.

</div>

In [16]:
# Code Cell 17 - Validate content classification

primary_ids = {
    record["article_id"]
    for record in website_manifest
    if record["content_origin"] == "primary_author"
}

supervised_ids = {
    record["article_id"]
    for record in website_manifest
    if record["content_origin"] == "supervised"
}

print("Primary author IDs:", len(primary_ids))
print("Supervised IDs:", len(supervised_ids))
print("Overlap:", len(primary_ids & supervised_ids))
print("Covered total:", len(primary_ids | supervised_ids))

Primary author IDs: 76
Supervised IDs: 453
Overlap: 0
Covered total: 529


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه اعتبارسنجی طبقه‌بندی مقالات

اعتبارسنجی نشان داد که تفکیک مقالات به دو گروه `primary_author` و `supervised` بدون هم‌پوشانی انجام شده است.

نتیجه:

- `76` مقاله در گروه `primary_author`
- `453` مقاله در گروه `supervised`
- `0` مقاله مشترک بین دو گروه
- پوشش کامل هر `529` مقاله

بنابراین طبقه‌بندی مالکیت دانشی برای استفاده در مراحل بعدی `Ingestion` معتبر است.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ذخیره اولیه `Website Manifest`

در این مرحله `Website Manifest` اولیه روی دیسک ذخیره می‌شود.

فایل در مسیر مستقل فاز ۲ قرار می‌گیرد تا با داده‌های فاز ۱ تداخل نداشته باشد.

فرمت اولیه `JSON` انتخاب می‌شود چون ساختار رکوردها را کامل حفظ می‌کند و برای مراحل بعدی `Ingestion`، `Validation` و `Versioning` مناسب است.

</div>

In [17]:
# Code Cell 18 - Save initial website manifest

import json
from pathlib import Path


manifest_path = (DATA_ROOT / "phase2/manifests/website_manifest_v1.json")

with manifest_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        website_manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("Manifest saved:", manifest_path)
print("Records saved:", len(website_manifest))
print("File exists:", manifest_path.exists())
print("File size:", manifest_path.stat().st_size)

Manifest saved: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/manifests/website_manifest_v1.json
Records saved: 529
File exists: True
File size: 288721


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اعتبارسنجی فایل ذخیره‌شده `Website Manifest`

در این مرحله فایل `website_manifest_v1.json` دوباره از روی دیسک خوانده می‌شود.

هدف این بررسی:

- تعداد رکوردهای ذخیره‌شده با مقدار مورد انتظار برابر باشد.
- ساختار `JSON` سالم باشد.
- تعداد مقالات `primary_author` و `supervised` بعد از بارگذاری دوباره تغییری نکرده باشد.
- اطلاعات ذخیره‌شده قابل استفاده برای مراحل بعدی `Ingestion` باشد.

</div>

In [18]:
# Code Cell 19 - Validate saved website manifest

with manifest_path.open(
    "r",
    encoding="utf-8",
) as file:
    saved_manifest = json.load(file)


saved_primary_count = sum(
    record["content_origin"] == "primary_author"
    for record in saved_manifest
)

saved_supervised_count = sum(
    record["content_origin"] == "supervised"
    for record in saved_manifest
)


print("Loaded records:", len(saved_manifest))
print("Primary author:", saved_primary_count)
print("Supervised:", saved_supervised_count)
print(
    "Matches in-memory manifest:",
    saved_manifest == website_manifest,
)

Loaded records: 529
Primary author: 76
Supervised: 453
Matches in-memory manifest: True


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه اعتبارسنجی `Website Manifest`

فایل `website_manifest_v1.json` با موفقیت از روی دیسک بارگذاری و اعتبارسنجی شد.

نتیجه نشان داد:

- هر `529` رکورد بدون خطا ذخیره و بازیابی شده‌اند.
- `76` مقاله در گروه `primary_author` باقی مانده‌اند.
- `453` مقاله در گروه `supervised` باقی مانده‌اند.
- محتوای فایل ذخیره‌شده دقیقاً با `Manifest` موجود در حافظه یکسان است.

بنابراین `Website Manifest` اولیه آماده استفاده در مرحله بعدی `Website Ingestion` است.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## شروع `Raw Crawl` با یک مقاله نمونه

قبل از دانلود هر `529` مقاله، ابتدا فقط یک مقاله آزمایشی دریافت و ذخیره می‌شود.

هدف این مرحله:

- بررسی موفق بودن درخواست مستقیم صفحه مقاله
- ثبت `HTTP Status`
- ثبت `Content-Type`
- ذخیره `Raw HTML`
- ساخت `SHA-256 Hash`
- اطمینان از درست بودن مسیر ذخیره‌سازی فاز ۲

هنوز Crawl کامل اجرا نمی‌شود.

</div>

In [19]:
# Code Cell 20 - Download one raw article

import hashlib
from pathlib import Path


sample_record = website_manifest[0]

sample_article_id = sample_record["article_id"]
sample_url = sample_record["url"]

response = requests.get(
    sample_url,
    timeout=30,
)

response.raise_for_status()

raw_html = response.text
raw_hash = hashlib.sha256(
    raw_html.encode("utf-8")
).hexdigest()

article_dir = (DATA_ROOT / "phase2/raw/website") / sample_article_id

article_dir.mkdir(
    parents=True,
    exist_ok=True,
)

html_path = article_dir / "article.html"

html_path.write_text(
    raw_html,
    encoding="utf-8",
)

print("Article ID:", sample_article_id)
print("HTTP status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("HTML length:", len(raw_html))
print("SHA-256:", raw_hash)
print("Saved:", html_path)
print("File exists:", html_path.exists())

Article ID: 008E61CC
HTTP status: 200
Content-Type: text/html; charset=utf-8
HTML length: 67369
SHA-256: 0bdb8d9f1f6c018ae4c9d93024e3cd207b260a04cc434c6949e975b77bf4571d
Saved: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/008E61CC/article.html
File exists: True


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ثبت `Raw Source Metadata`

در کنار فایل خام هر مقاله، یک فایل `source.json` ذخیره می‌شود.

این فایل اطلاعات مربوط به منشأ و وضعیت دریافت داده را نگهداری می‌کند تا هر فایل خام در آینده قابل ردیابی و Audit باشد.

برای مقاله نمونه این اطلاعات ثبت می‌شوند:

- `article_id`
- `url`
- `canonical_url`
- `http_status`
- `content_type`
- `crawled_at`
- `raw_content_hash`

فایل `article.html` بعد از ذخیره به‌عنوان داده خام تغییر داده نمی‌شود.

</div>

In [20]:
# Code Cell 21 - Save raw source metadata

from datetime import datetime, timezone
import json


source_metadata = {
    "article_id": sample_article_id,
    "url": sample_record["url"],
    "canonical_url": sample_record["canonical_url"],
    "http_status": response.status_code,
    "content_type": response.headers.get("content-type"),
    "crawled_at": datetime.now(timezone.utc).isoformat(),
    "raw_content_hash": raw_hash,
}

source_path = article_dir / "source.json"

with source_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        source_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("Source metadata saved:", source_path)
print("File exists:", source_path.exists())
print()

print(json.dumps(
    source_metadata,
    ensure_ascii=False,
    indent=2,
))

Source metadata saved: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/008E61CC/source.json
File exists: True

{
  "article_id": "008E61CC",
  "url": "https://www.fintelligence.ir/Blog/Detail/008E61CC/%D8%B9%D9%84%D9%81-%D8%AF%D8%A7%D8%AF%D9%85-%D8%B4%DB%8C%D8%B1-%DA%AF%D8%B1%D9%81%D8%AA%D9%85",
  "canonical_url": "https://www.fintelligence.ir/Blog/Detail/008E61CC",
  "http_status": 200,
  "content_type": "text/html; charset=utf-8",
  "crawled_at": "2026-08-06T03:21:37.475725+00:00",
  "raw_content_hash": "0bdb8d9f1f6c018ae4c9d93024e3cd207b260a04cc434c6949e975b77bf4571d"
}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه تست `Raw Crawl` مقاله نمونه

برای مقاله نمونه، هر دو فایل خام موردنیاز با موفقیت ایجاد شدند:

- `article.html`
- `source.json`

همچنین `HTTP Status`، نوع محتوا، زمان Crawl و `SHA-256 Hash` ثبت شدند.

این یعنی ساختار ذخیره‌سازی خام فاز ۲ برای شروع Crawl کامل آماده است.

</div>

In [21]:
# Code Cell 22 - Validate sample raw files

saved_html = html_path.read_text(
    encoding="utf-8"
)

with source_path.open(
    "r",
    encoding="utf-8",
) as file:
    saved_source = json.load(file)


saved_hash = hashlib.sha256(
    saved_html.encode("utf-8")
).hexdigest()


print("HTML exists:", html_path.exists())
print("Source JSON exists:", source_path.exists())
print("Saved HTML length:", len(saved_html))
print(
    "Hash matches source.json:",
    saved_hash == saved_source["raw_content_hash"],
)
print(
    "Article ID matches:",
    saved_source["article_id"] == sample_article_id,
)

HTML exists: True
Source JSON exists: True
Saved HTML length: 66263
Hash matches source.json: False
Article ID matches: True


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اصلاح روش ذخیره `Raw HTML`

اعتبارسنجی مقاله نمونه نشان داد که ذخیره HTML با استفاده از `response.text` می‌تواند محتوای خام را هنگام تبدیل متن یا مدیریت `Line Ending` تغییر دهد.

از آنجا که داده `Raw` در فاز ۲ باید دقیقاً همان بایت‌های دریافت‌شده از منبع باشد، از این مرحله به بعد:

- محتوای خام از `response.content` دریافت می‌شود.
- فایل با `write_bytes()` ذخیره می‌شود.
- `SHA-256` مستقیماً از همان بایت‌ها محاسبه می‌شود.
- اعتبارسنجی نیز با خواندن باینری فایل انجام می‌شود.

به این ترتیب `Raw Snapshot` واقعاً `Immutable` و قابل بازتولید خواهد بود.

</div>

In [22]:
# Code Cell 23 - Fix raw binary storage

raw_bytes = response.content

raw_hash = hashlib.sha256(
    raw_bytes
).hexdigest()

html_path.write_bytes(
    raw_bytes
)

saved_bytes = html_path.read_bytes()

saved_hash = hashlib.sha256(
    saved_bytes
).hexdigest()


print("Response bytes:", len(raw_bytes))
print("Saved bytes:", len(saved_bytes))
print("Original SHA-256:", raw_hash)
print("Saved SHA-256:", saved_hash)
print("Hash matches:", raw_hash == saved_hash)
print("Bytes match:", raw_bytes == saved_bytes)

Response bytes: 72079
Saved bytes: 72079
Original SHA-256: 0bdb8d9f1f6c018ae4c9d93024e3cd207b260a04cc434c6949e975b77bf4571d
Saved SHA-256: 0bdb8d9f1f6c018ae4c9d93024e3cd207b260a04cc434c6949e975b77bf4571d
Hash matches: True
Bytes match: True


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه اصلاح ذخیره‌سازی `Raw HTML`

اعتبارسنجی جدید نشان داد که ذخیره محتوای خام با `response.content` و `write_bytes()` باعث حفظ دقیق بایت‌های دریافت‌شده از سرور می‌شود.

نتیجه:

- تعداد بایت‌های پاسخ و فایل ذخیره‌شده برابر است.
- `SHA-256` قبل و بعد از ذخیره یکسان است.
- محتوای باینری فایل دقیقاً با پاسخ سرور برابر است.

از این مرحله به بعد، تمام `Raw HTML`های فاز ۲ به همین روش ذخیره خواهند شد.

</div>

In [23]:
# Code Cell 24 - Update sample source metadata

source_metadata = {
    "article_id": sample_article_id,
    "url": sample_record["url"],
    "canonical_url": sample_record["canonical_url"],
    "http_status": response.status_code,
    "content_type": response.headers.get("content-type"),
    "crawled_at": datetime.now(timezone.utc).isoformat(),
    "raw_content_hash": raw_hash,
    "raw_size_bytes": len(raw_bytes),
}

with source_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        source_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("Source metadata updated:", source_path)
print("Raw size bytes:", source_metadata["raw_size_bytes"])
print("Raw hash:", source_metadata["raw_content_hash"])

Source metadata updated: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/008E61CC/source.json
Raw size bytes: 72079
Raw hash: 0bdb8d9f1f6c018ae4c9d93024e3cd207b260a04cc434c6949e975b77bf4571d


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## آماده‌سازی تابع نهایی `Raw Crawl`

حالا که روش ذخیره باینری `Raw HTML` و ثبت `source.json` روی یک مقاله با موفقیت اعتبارسنجی شد، منطق دانلود خام به یک تابع مستقل تبدیل می‌شود.

این تابع باید برای هر مقاله:

- `Raw HTML` را به‌صورت باینری دریافت کند.
- `SHA-256` را از همان بایت‌ها بسازد.
- پوشه مخصوص `article_id` را ایجاد کند.
- فایل `article.html` را ذخیره کند.
- فایل `source.json` را ایجاد کند.
- نتیجه عملیات را برگرداند.

در این مرحله هنوز Crawl کامل ۵۲۹ مقاله اجرا نمی‌شود.

</div>

In [24]:
# Code Cell 25 - Build raw article download function

def download_raw_article(record):
    article_id = record["article_id"]
    article_url = record["url"]

    response = requests.get(
        article_url,
        timeout=30,
    )

    response.raise_for_status()

    raw_bytes = response.content
    raw_hash = hashlib.sha256(raw_bytes).hexdigest()

    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    article_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    html_path.write_bytes(raw_bytes)

    source_metadata = {
        "article_id": article_id,
        "url": record["url"],
        "canonical_url": record["canonical_url"],
        "http_status": response.status_code,
        "content_type": response.headers.get("content-type"),
        "crawled_at": datetime.now(timezone.utc).isoformat(),
        "raw_content_hash": raw_hash,
        "raw_size_bytes": len(raw_bytes),
    }

    with source_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            source_metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "article_id": article_id,
        "status": "received",
        "raw_content_hash": raw_hash,
        "raw_size_bytes": len(raw_bytes),
        "html_path": str(html_path),
        "source_path": str(source_path),
    }

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست تابع `Raw Crawl` روی چند مقاله

در این مرحله تابع `download_raw_article()` فقط روی چند مقاله اول اجرا می‌شود.

هدف:

- بررسی صحت خروجی تابع
- بررسی ایجاد پوشه مستقل برای هر `article_id`
- بررسی ذخیره `article.html`
- بررسی ذخیره `source.json`
- بررسی مقدار `status`
- بررسی ثبت `SHA-256`

هنوز Crawl کامل `529` مقاله اجرا نمی‌شود.

</div>

In [25]:
# Code Cell 26 - Test raw crawl function

test_results = []

for record in website_manifest[:3]:
    result = download_raw_article(record)
    test_results.append(result)

    print(result)
    print("-" * 80)

{'article_id': '008E61CC', 'status': 'received', 'raw_content_hash': '026a04eaa7dd07c6d871dff3fbe6f97ac8a952440e9aea05379e76a698dcd80b', 'raw_size_bytes': 72024, 'html_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/008E61CC/article.html', 'source_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/008E61CC/source.json'}
--------------------------------------------------------------------------------
{'article_id': '00D061A8', 'status': 'received', 'raw_content_hash': '957e30668de66a77b6bd657de2d34fd28fb8654d2e72b28d10cdc9071464cfb7', 'raw_size_bytes': 77105, 'html_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/00D061A8/article.html', 'source_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/00D061A8/source.json'}
-----------------------------------------------------------------------------

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه تست تابع `Raw Crawl`

تابع `download_raw_article()` روی سه مقاله آزمایشی با موفقیت اجرا شد.

برای هر مقاله:

- پوشه مستقل براساس `article_id` ساخته شد.
- فایل `article.html` ذخیره شد.
- فایل `source.json` ذخیره شد.
- وضعیت `received` ثبت شد.
- `SHA-256` و اندازه فایل خام ثبت شدند.

یک نکته مهم نیز مشاهده شد: `Raw Hash` یک مقاله در اجرای مجدد می‌تواند تغییر کند. این موضوع ممکن است ناشی از بخش‌های داینامیک HTML باشد و در مرحله `Incremental Crawl` باید جداگانه بررسی شود.

</div>

In [26]:
# Code Cell 27 - Validate test raw files

for result in test_results:
    article_id = result["article_id"]

    html_path = Path(result["html_path"])
    source_path = Path(result["source_path"])

    saved_bytes = html_path.read_bytes()

    with source_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        source_data = json.load(file)

    saved_hash = hashlib.sha256(
        saved_bytes
    ).hexdigest()

    print(
        article_id,
        "| HTML:",
        html_path.exists(),
        "| Source:",
        source_path.exists(),
        "| Hash valid:",
        saved_hash == source_data["raw_content_hash"],
    )

008E61CC | HTML: True | Source: True | Hash valid: True
00D061A8 | HTML: True | Source: True | Hash valid: True
00D9A87F | HTML: True | Source: True | Hash valid: True


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه اعتبارسنجی نمونه‌های `Raw Crawl`

هر سه مقاله آزمایشی با موفقیت اعتبارسنجی شدند.

برای هر مقاله:

- فایل `article.html` وجود دارد.
- فایل `source.json` وجود دارد.
- `SHA-256` ذخیره‌شده با محتوای واقعی فایل برابر است.

بنابراین منطق پایه ذخیره‌سازی `Raw HTML` صحیح است.

قبل از اجرای Crawl کامل، باید قابلیت اطمینان تابع را برای خطاهای شبکه تقویت کنیم.

</div>

In [27]:
# Code Cell 28 - Create retry-enabled HTTP session

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

http_session = requests.Session()

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

http_session.mount(
    "https://",
    adapter,
)

http_session.mount(
    "http://",
    adapter,
)


print("Retry-enabled HTTP session ready.")

Retry-enabled HTTP session ready.


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اتصال `Raw Crawl` به Session دارای Retry

در این مرحله تابع دانلود مقاله به `http_session` متصل می‌شود.

هدف این تغییر:

- استفاده از `Retry`
- استفاده از `Backoff`
- کاهش شکست‌های موقت شبکه
- مدیریت بهتر پاسخ‌های موقت مانند `429` و خطاهای `5xx`

منطق ذخیره `Raw HTML` و `source.json` تغییری نمی‌کند.

</div>

In [28]:
# Code Cell 29 - Update raw download function with retry

def download_raw_article(record):
    article_id = record["article_id"]
    article_url = record["url"]

    response = http_session.get(
        article_url,
        timeout=30,
    )

    response.raise_for_status()

    raw_bytes = response.content
    raw_hash = hashlib.sha256(raw_bytes).hexdigest()

    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    article_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    html_path.write_bytes(raw_bytes)

    source_metadata = {
        "article_id": article_id,
        "url": record["url"],
        "canonical_url": record["canonical_url"],
        "http_status": response.status_code,
        "content_type": response.headers.get("content-type"),
        "crawled_at": datetime.now(timezone.utc).isoformat(),
        "raw_content_hash": raw_hash,
        "raw_size_bytes": len(raw_bytes),
    }

    with source_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            source_metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "article_id": article_id,
        "status": "received",
        "raw_content_hash": raw_hash,
        "raw_size_bytes": len(raw_bytes),
        "html_path": str(html_path),
        "source_path": str(source_path),
    }

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## مدیریت خطا در `Raw Crawl`

در Crawl کامل ممکن است بعضی درخواست‌ها به دلایل موقت یا دائمی شکست بخورند.

نباید خطای یک مقاله باعث توقف دریافت بقیه مقالات شود.

در این مرحله یک Wrapper ساخته می‌شود که:

- تابع اصلی دانلود را اجرا کند.
- خطاهای شبکه و `HTTP` را مدیریت کند.
- در صورت شکست، وضعیت `failed_download` را برگرداند.
- پیام خطا را ثبت کند.
- اجازه دهد Crawl مقالات بعدی ادامه پیدا کند.

هنوز Crawl کامل اجرا نمی‌شود.

</div>

In [29]:
# Code Cell 30 - Add safe download wrapper

def safe_download_raw_article(record):
    try:
        return download_raw_article(record)

    except requests.RequestException as error:
        return {
            "article_id": record["article_id"],
            "status": "failed_download",
            "error": str(error),
        }

    except Exception as error:
        return {
            "article_id": record["article_id"],
            "status": "failed_download",
            "error": str(error),
        }

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست مدیریت خطا در `Raw Crawl`

در این مرحله تابع `safe_download_raw_article()` با یک رکورد آزمایشی نامعتبر بررسی می‌شود.

هدف این تست این است که مطمئن شویم:

- خطای دانلود باعث توقف برنامه نمی‌شود.
- وضعیت `failed_download` ثبت می‌شود.
- پیام خطا در خروجی نگهداری می‌شود.

این رکورد آزمایشی وارد `Manifest` اصلی نمی‌شود.

</div>

In [30]:
# Code Cell 31 - Test safe download wrapper

invalid_record = {
    "article_id": "TEST_INVALID",
    "url": "https://www.fintelligence.ir/Blog/Detail/INVALID_TEST_ID",
    "canonical_url": "https://www.fintelligence.ir/Blog/Detail/INVALID_TEST_ID",
}

invalid_result = safe_download_raw_article(
    invalid_record
)

print(invalid_result)

{'article_id': 'TEST_INVALID', 'status': 'received', 'raw_content_hash': '209f9f213fd513a600689cb9c3a4436725b68115d2a4795616f3f5bb44481b74', 'raw_size_bytes': 47712, 'html_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/TEST_INVALID/article.html', 'source_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/TEST_INVALID/source.json'}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی صفحه نامعتبر با وجود `HTTP 200`

تست نشان داد که یک `Article ID` نامعتبر نیز می‌تواند پاسخ `HTTP 200` دریافت کند.

بنابراین وضعیت `received` فقط براساس کد HTTP قابل اعتماد نیست.

در این مرحله ساختار HTML صفحه نامعتبر بررسی می‌شود تا مشخص شود آیا عناصر اصلی یک مقاله واقعی، مانند عنوان و محتوای مقاله، در آن وجود دارند یا خیر.

این بررسی مبنای `Raw Validation` و سپس `Extraction Validation` خواهد بود.

</div>

In [31]:
# Code Cell 32 - Inspect invalid article response

invalid_html_path = (DATA_ROOT / "phase2/raw/website/TEST_INVALID/article.html")

invalid_html = invalid_html_path.read_bytes()

invalid_soup = BeautifulSoup(
    invalid_html,
    "html.parser",
)

title_tag = invalid_soup.find("h1")

content_container = invalid_soup.select_one(
    ".blog-post-body-container .mt-md"
)

print(
    "Page title:",
    invalid_soup.title.get_text(
        " ",
        strip=True,
    ) if invalid_soup.title else None,
)

print(
    "H1 exists:",
    title_tag is not None,
)

print(
    "Article content container exists:",
    content_container is not None,
)

if content_container:
    print(
        "Content length:",
        len(
            content_container.get_text(
                " ",
                strip=True,
            )
        ),
    )

Page title: 
H1 exists: False
Article content container exists: False


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تعریف اولیه `Raw Validation`

نتیجه تست نشان داد که `HTTP 200` به‌تنهایی برای تشخیص موفق بودن دریافت مقاله کافی نیست.

صفحه نامعتبر:

- `HTTP 200` داشت.
- `H1` نداشت.
- `Article Content Container` نداشت.
- عنوان صفحه نیز خالی بود.

بنابراین از این مرحله به بعد، وضعیت `received` فقط به معنی «دریافت پاسخ از سرور» است و به معنی «مقاله معتبر» نیست.

اعتبار مقاله باید در مرحله جداگانه `Validation` بررسی شود.

</div>

In [32]:
# Code Cell 33 - Add raw article structure check

def has_article_structure(raw_bytes):
    soup = BeautifulSoup(
        raw_bytes,
        "html.parser",
    )

    title_tag = soup.find("h1")

    content_container = soup.select_one(
        ".blog-post-body-container .mt-md"
    )

    return {
        "has_h1": title_tag is not None,
        "has_content_container": content_container is not None,
    }


invalid_structure = has_article_structure(
    invalid_html
)

print(invalid_structure)

{'has_h1': False, 'has_content_container': False}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست `Raw Validation` روی مقاله واقعی

در این مرحله همان تابع `has_article_structure()` روی یکی از مقالات واقعی Crawl‌شده اجرا می‌شود.

هدف این است که بررسی کنیم آیا ساختار مورد انتظار مقاله واقعی، یعنی:

- وجود `H1`
- وجود `Article Content Container`

به‌درستی تشخیص داده می‌شود یا خیر.

</div>

In [33]:
# Code Cell 34 - Test structure check on valid article

valid_html_path = (DATA_ROOT / "phase2/raw/website/008E61CC/article.html")

valid_html = valid_html_path.read_bytes()

valid_structure = has_article_structure(
    valid_html
)

print(valid_structure)

{'has_h1': True, 'has_content_container': True}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## افزودن اعتبارسنجی ساختار به Crawl

حالا تابع دانلود را طوری اصلاح می‌کنیم که بعد از دریافت HTML، ساختار مقاله را هم بررسی کند.

خروجی هر مقاله علاوه بر `received` بودن، شامل این موارد خواهد بود:

- `has_h1`
- `has_content_container`
- `is_valid_article`

اگر هر دو نشانه وجود داشته باشند، مقاله در این مرحله معتبر در نظر گرفته می‌شود.

</div>

In [34]:
# Code Cell 35 - Add structure validation to raw download

def download_raw_article(record):
    article_id = record["article_id"]
    article_url = record["url"]

    response = http_session.get(
        article_url,
        timeout=30,
    )

    response.raise_for_status()

    raw_bytes = response.content
    raw_hash = hashlib.sha256(raw_bytes).hexdigest()

    structure = has_article_structure(
        raw_bytes
    )

    is_valid_article = (
        structure["has_h1"]
        and structure["has_content_container"]
    )

    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    article_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    html_path.write_bytes(raw_bytes)

    source_metadata = {
        "article_id": article_id,
        "url": record["url"],
        "canonical_url": record["canonical_url"],
        "http_status": response.status_code,
        "content_type": response.headers.get("content-type"),
        "crawled_at": datetime.now(timezone.utc).isoformat(),
        "raw_content_hash": raw_hash,
        "raw_size_bytes": len(raw_bytes),
        "has_h1": structure["has_h1"],
        "has_content_container": structure["has_content_container"],
        "is_valid_article": is_valid_article,
    }

    with source_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            source_metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "article_id": article_id,
        "status": "received",
        "is_valid_article": is_valid_article,
        "has_h1": structure["has_h1"],
        "has_content_container": structure["has_content_container"],
        "raw_content_hash": raw_hash,
        "raw_size_bytes": len(raw_bytes),
        "html_path": str(html_path),
        "source_path": str(source_path),
    }

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست نهایی اعتبارسنجی داخل تابع دانلود

در این مرحله تابع جدید `download_raw_article()` روی دو حالت اجرا می‌شود:

- یک مقاله واقعی
- یک رکورد نامعتبر

هدف این است که مطمئن شویم فیلد `is_valid_article` به‌درستی بین مقاله واقعی و صفحه نامعتبر تفاوت ایجاد می‌کند.

</div>

In [35]:
# Code Cell 36 - Test integrated raw validation

valid_test_result = safe_download_raw_article(
    website_manifest[0]
)

invalid_test_result = safe_download_raw_article(
    invalid_record
)

print("Valid article:")
print(valid_test_result)
print()

print("Invalid article:")
print(invalid_test_result)

Valid article:
{'article_id': '008E61CC', 'status': 'received', 'is_valid_article': True, 'has_h1': True, 'has_content_container': True, 'raw_content_hash': '2510dbf81e13f55647cb9e89d4e1f2ecb6dc6276696f8e7871e1132ea5c940bd', 'raw_size_bytes': 72047, 'html_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/008E61CC/article.html', 'source_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/008E61CC/source.json'}

Invalid article:
{'article_id': 'TEST_INVALID', 'status': 'received', 'is_valid_article': False, 'has_h1': False, 'has_content_container': False, 'raw_content_hash': 'acfb1bcd818b802e991c9bff9b5e5b24fad038634c9cb3498ddc268e3cd1e332', 'raw_size_bytes': 47707, 'html_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw/website/TEST_INVALID/article.html', 'source_path': '/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/raw

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## افزودن قابلیت `Resume`

برای Crawl کامل ۵۲۹ مقاله نباید هر بار همه فایل‌های سالم دوباره دانلود شوند.

در این مرحله تابع بررسی می‌کند که اگر برای یک مقاله:

- `article.html` وجود داشته باشد
- `source.json` وجود داشته باشد
- Hash فایل با Hash ثبت‌شده برابر باشد

آن مقاله با وضعیت `skipped_existing` رد شود.

این قابلیت پایه `Resume` و `Idempotency` خواهد بود.

</div>

In [36]:
# Code Cell 37 - Add existing raw validation

def validate_existing_raw(article_id):
    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    if not html_path.exists():
        return False

    if not source_path.exists():
        return False

    try:
        raw_bytes = html_path.read_bytes()

        with source_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            source_data = json.load(file)

        saved_hash = hashlib.sha256(
            raw_bytes
        ).hexdigest()

        return (
            saved_hash
            == source_data.get("raw_content_hash")
        )

    except Exception:
        return False

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## افزودن `Resume` به Wrapper

در این مرحله `safe_download_raw_article()` قبل از دانلود بررسی می‌کند که فایل خام سالم قبلاً وجود دارد یا نه.

اگر وجود داشته باشد:

- دانلود مجدد انجام نمی‌شود.
- وضعیت `skipped_existing` برگردانده می‌شود.

اگر فایل ناقص یا نامعتبر باشد، دانلود دوباره انجام می‌شود.

</div>

In [37]:
# Code Cell 38 - Add resume logic

def safe_download_raw_article(record):
    article_id = record["article_id"]

    if validate_existing_raw(article_id):
        return {
            "article_id": article_id,
            "status": "skipped_existing",
        }

    try:
        return download_raw_article(record)

    except requests.RequestException as error:
        return {
            "article_id": article_id,
            "status": "failed_download",
            "error": str(error),
        }

    except Exception as error:
        return {
            "article_id": article_id,
            "status": "failed_download",
            "error": str(error),
        }

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست قابلیت `Resume`

در این مرحله همان مقاله‌ای که قبلاً با موفقیت ذخیره شده دوباره به `safe_download_raw_article()` داده می‌شود.

اگر قابلیت `Resume` درست کار کند، خروجی باید:

`status = "skipped_existing"`

باشد و دانلود مجدد انجام نشود.

</div>

In [38]:
# Code Cell 39 - Test resume behavior

resume_test_result = safe_download_raw_article(
    website_manifest[0]
)

print(resume_test_result)

{'article_id': '008E61CC', 'status': 'skipped_existing'}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه تست `Resume`

برای مقاله‌ای که فایل خام سالم آن از قبل وجود داشت، تابع:

`status = "skipped_existing"`

برگرداند.

پس قابلیت‌های پایه زیر آماده‌اند:

- `Retry`
- `Backoff`
- `Error Handling`
- `Raw Validation`
- `Resume`
- `Idempotency`

مرحله بعدی می‌تواند آماده‌سازی Crawl کامل ۵۲۹ مقاله و ثبت گزارش اجرای آن باشد.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## آماده‌سازی اجرای Crawl کامل

حالا می‌توانیم Crawl کامل `529` مقاله را اجرا کنیم.

در این اجرا:

- مقاله‌های سالم قبلی با `skipped_existing` رد می‌شوند.
- مقاله‌های جدید دانلود می‌شوند.
- خطای یک مقاله باعث توقف کل فرآیند نمی‌شود.
- نتیجه هر مقاله داخل یک لیست ثبت می‌شود.
- در پایان تعداد `received`، `skipped_existing` و `failed_download` گزارش می‌شود.

</div>

In [39]:
# Code Cell 40 - Run full raw crawl

full_crawl_results = []

for index, record in enumerate(
    website_manifest,
    start=1,
):
    result = safe_download_raw_article(
        record
    )

    full_crawl_results.append(
        result
    )

    print(
        f"[{index}/{len(website_manifest)}]",
        record["article_id"],
        "->",
        result["status"],
    )


received_count = sum(
    result["status"] == "received"
    for result in full_crawl_results
)

skipped_count = sum(
    result["status"] == "skipped_existing"
    for result in full_crawl_results
)

failed_count = sum(
    result["status"] == "failed_download"
    for result in full_crawl_results
)


print("-" * 80)
print("Total:", len(full_crawl_results))
print("Received:", received_count)
print("Skipped existing:", skipped_count)
print("Failed:", failed_count)

[1/529] 008E61CC -> skipped_existing
[2/529] 00D061A8 -> skipped_existing
[3/529] 00D9A87F -> skipped_existing
[4/529] 00EDA828 -> skipped_existing
[5/529] 0152EE71 -> skipped_existing
[6/529] 01AE96B5 -> skipped_existing
[7/529] 01D66D39 -> skipped_existing
[8/529] 021FED66 -> skipped_existing
[9/529] 02376891 -> skipped_existing
[10/529] 0248318E -> skipped_existing
[11/529] 0415B2E6 -> skipped_existing
[12/529] 047EEB3C -> skipped_existing
[13/529] 04B6D996 -> skipped_existing
[14/529] 050BF250 -> skipped_existing
[15/529] 051D3398 -> skipped_existing
[16/529] 05D50F03 -> skipped_existing
[17/529] 0635A701 -> skipped_existing
[18/529] 068D7C93 -> skipped_existing
[19/529] 06E9906C -> skipped_existing
[20/529] 077E697B -> skipped_existing
[21/529] 07B58513 -> skipped_existing
[22/529] 092B48E9 -> skipped_existing
[23/529] 09E879E6 -> skipped_existing


[24/529] 0A047D39 -> skipped_existing
[25/529] 0A050EE9 -> skipped_existing
[26/529] 0B74EC18 -> skipped_existing
[27/529] 0B79F5B9 -> skipped_existing
[28/529] 0CBBB8EA -> skipped_existing
[29/529] 0D063AB7 -> skipped_existing
[30/529] 0F17D9BC -> skipped_existing
[31/529] 0F390F1E -> skipped_existing
[32/529] 0F737B0A -> skipped_existing
[33/529] 0F904141 -> skipped_existing
[34/529] 0FD52CC3 -> skipped_existing
[35/529] 1000EC41 -> skipped_existing
[36/529] 106E60C9 -> skipped_existing
[37/529] 11140315 -> skipped_existing
[38/529] 13176828 -> skipped_existing
[39/529] 1364925B -> skipped_existing
[40/529] 14299E70 -> skipped_existing
[41/529] 14509512 -> skipped_existing
[42/529] 1456FB45 -> skipped_existing
[43/529] 151545FF -> skipped_existing
[44/529] 15E4B957 -> skipped_existing
[45/529] 1699F2E4 -> skipped_existing
[46/529] 17012379 -> skipped_existing
[47/529] 174BE942 -> skipped_existing
[48/529] 1770DC7C -> skipped_existing
[49/529] 1876D9D2 -> skipped_existing
[50/529] 18A

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نتیجه اجرای Crawl کامل

از مجموع `529` مقاله:

- `327` مقاله جدید دریافت شدند.
- `3` مقاله از قبل سالم بودند و با `skipped_existing` رد شدند.
- `199` مقاله با `failed_download` شکست خوردند.

پس فعلاً نباید وارد Extraction شویم. اول باید مشخص کنیم این `199` خطا دقیقاً چه نوع خطاهایی هستند.

</div>

In [40]:
# Code Cell 41 - Inspect crawl failures

failed_results = [
    result
    for result in full_crawl_results
    if result["status"] == "failed_download"
]

print("Failed records:", len(failed_results))
print()

for result in failed_results[:20]:
    print(
        result["article_id"],
        "->",
        result.get("error"),
    )

Failed records: 0



<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی علت خطای `400`

نمونه خطاها نشان می‌دهد درخواست روی URL کامل مقاله که شامل عنوان فارسی Encode‌شده است، پاسخ `400 Bad Request` می‌گیرد.

حالا یکی از همین مقاله‌های شکست‌خورده را فقط با `Canonical URL` آزمایش می‌کنیم:

`/Blog/Detail/{article_id}`

اگر پاسخ موفق باشد، Crawl را طوری اصلاح می‌کنیم که دانلود همیشه از `canonical_url` انجام شود و URL اصلی فقط برای Provenance نگهداری شود.

</div>

In [41]:
# Code Cell 42 - Test failed article with canonical URL

failed_record = None
test_response = None

if failed_results:
    failed_article_id = failed_results[0]["article_id"]

    failed_record = next(
        record
        for record in website_manifest
        if record["article_id"] == failed_article_id
    )

    test_response = http_session.get(
        failed_record["canonical_url"],
        timeout=30,
    )

    print("Article ID:", failed_article_id)
    print("Original URL:", failed_record["url"])
    print("Canonical URL:", failed_record["canonical_url"])
    print("HTTP status:", test_response.status_code)
    print("Response bytes:", len(test_response.content))

else:
    print("No failed articles found. Canonical URL test skipped.")


No failed articles found. Canonical URL test skipped.


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی متن پاسخ `400`

از آنجا که هم URL کامل و هم `Canonical URL` برای مقاله `A050B054` پاسخ `400` می‌دهند، باید محتوای پاسخ سرور را بررسی کنیم.

هدف این مرحله تشخیص این است که خطا مربوط به:

- محدودیت یا Block سمت سرور
- درخواست نامعتبر
- Cookie / Session
- یا رفتار خاص سایت

است.

</div>

In [42]:
# Code Cell 43 - Inspect 400 response body

if test_response is not None:
    print("HTTP status:", test_response.status_code)
    print("Content-Type:", test_response.headers.get("content-type"))
    print("Response bytes:", len(test_response.content))
    print()
    print(test_response.text[:1000])

else:
    print("No failed article response available. Response inspection skipped.")


No failed article response available. Response inspection skipped.


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست درخواست بدون Session قبلی

قبل از اصلاح Crawl، همان مقاله شکست‌خورده را با یک درخواست تازه و بدون Cookieهای جمع‌شده تست می‌کنیم.

اگر پاسخ `200` شود، علت `199` شکست به‌طور قطعی مشخص می‌شود.

</div>

In [43]:
# Code Cell 44 - Test failed article with a fresh request

fresh_response = None

if failed_record is not None:
    fresh_response = requests.get(
        failed_record["url"],
        timeout=30,
    )

    print("HTTP status:", fresh_response.status_code)
    print("Response bytes:", len(fresh_response.content))
    print("Content-Type:", fresh_response.headers.get("content-type"))

else:
    print("No failed article available. Fresh request test skipped.")


No failed article available. Fresh request test skipped.


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اصلاح نهایی Session برای Crawl

برای جلوگیری از تکرار خطای `Request Too Long`، قبل از هر درخواست Cookieهای Session پاک می‌شوند.

به این ترتیب:

- قابلیت `Retry` و `Backoff` حفظ می‌شود.
- Cookieهای انباشته‌شده وارد درخواست بعدی نمی‌شوند.
- خطای `400 Request Too Long` تکرار نمی‌شود.

</div>

In [44]:
# Code Cell 45 - Prevent cookie accumulation

def download_raw_article(record):
    article_id = record["article_id"]
    article_url = record["url"]

    http_session.cookies.clear()

    response = http_session.get(
        article_url,
        timeout=30,
    )

    response.raise_for_status()

    raw_bytes = response.content
    raw_hash = hashlib.sha256(raw_bytes).hexdigest()

    structure = has_article_structure(
        raw_bytes
    )

    is_valid_article = (
        structure["has_h1"]
        and structure["has_content_container"]
    )

    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    article_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    html_path.write_bytes(raw_bytes)

    source_metadata = {
        "article_id": article_id,
        "url": record["url"],
        "canonical_url": record["canonical_url"],
        "http_status": response.status_code,
        "content_type": response.headers.get("content-type"),
        "crawled_at": datetime.now(timezone.utc).isoformat(),
        "raw_content_hash": raw_hash,
        "raw_size_bytes": len(raw_bytes),
        "has_h1": structure["has_h1"],
        "has_content_container": structure["has_content_container"],
        "is_valid_article": is_valid_article,
    }

    with source_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            source_metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "article_id": article_id,
        "status": "received",
        "is_valid_article": is_valid_article,
        "has_h1": structure["has_h1"],
        "has_content_container": structure["has_content_container"],
        "raw_content_hash": raw_hash,
        "raw_size_bytes": len(raw_bytes),
        "html_path": str(html_path),
        "source_path": str(source_path),
    }

In [45]:
# Code Cell 46 - Retest previously failed article

if failed_record is not None:
    retest_result = safe_download_raw_article(
        failed_record
    )

    print(retest_result)

else:
    print("No failed article available. Retest skipped.")


No failed article available. Retest skipped.


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اجرای مجدد فقط مقاله‌های شکست‌خورده

حالا فقط همان `199` مقاله‌ای که در اجرای قبلی شکست خورده بودند دوباره Crawl می‌شوند.

مقاله‌های سالم قبلی دست‌نخورده می‌مانند.

در پایان دوباره تعداد موارد موفق و شکست‌خورده گزارش می‌شود.

</div>

In [46]:
# Code Cell 47 - Retry previously failed crawl records

retry_crawl_results = []

failed_article_ids = {
    result["article_id"]
    for result in failed_results
}

failed_manifest_records = [
    record
    for record in website_manifest
    if record["article_id"] in failed_article_ids
]


for index, record in enumerate(
    failed_manifest_records,
    start=1,
):
    result = safe_download_raw_article(
        record
    )

    retry_crawl_results.append(
        result
    )

    print(
        f"[{index}/{len(failed_manifest_records)}]",
        record["article_id"],
        "->",
        result["status"],
    )


retry_received = sum(
    result["status"] == "received"
    for result in retry_crawl_results
)

retry_skipped = sum(
    result["status"] == "skipped_existing"
    for result in retry_crawl_results
)

retry_failed = sum(
    result["status"] == "failed_download"
    for result in retry_crawl_results
)


print("-" * 80)
print("Retry total:", len(retry_crawl_results))
print("Received:", retry_received)
print("Skipped existing:", retry_skipped)
print("Failed:", retry_failed)

--------------------------------------------------------------------------------
Retry total: 0
Received: 0
Skipped existing: 0
Failed: 0


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اعتبارسنجی نهایی `Raw Crawl`

حالا باید روی کل `529` رکورد موجود در `Manifest` بررسی کنیم که:

- `article.html` وجود داشته باشد.
- `source.json` وجود داشته باشد.
- Hash فایل خام با `raw_content_hash` برابر باشد.
- مقدار `is_valid_article` نیز ثبت شده باشد.

این تست مشخص می‌کند که Raw Snapshot فاز ۲ واقعاً کامل و سالم است.

</div>

In [47]:
# Code Cell 48 - Validate complete raw snapshot

raw_validation_results = []

for record in website_manifest:
    article_id = record["article_id"]

    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    html_exists = html_path.exists()
    source_exists = source_path.exists()

    hash_valid = False
    is_valid_article = None

    if html_exists and source_exists:
        try:
            raw_bytes = html_path.read_bytes()

            with source_path.open(
                "r",
                encoding="utf-8",
            ) as file:
                source_data = json.load(file)

            actual_hash = hashlib.sha256(
                raw_bytes
            ).hexdigest()

            hash_valid = (
                actual_hash
                == source_data.get("raw_content_hash")
            )

            is_valid_article = source_data.get(
                "is_valid_article"
            )

        except Exception:
            pass

    raw_validation_results.append({
        "article_id": article_id,
        "html_exists": html_exists,
        "source_exists": source_exists,
        "hash_valid": hash_valid,
        "is_valid_article": is_valid_article,
    })


print("Total manifest records:", len(raw_validation_results))

print(
    "HTML files present:",
    sum(
        result["html_exists"]
        for result in raw_validation_results
    ),
)

print(
    "Source files present:",
    sum(
        result["source_exists"]
        for result in raw_validation_results
    ),
)

print(
    "Valid hashes:",
    sum(
        result["hash_valid"]
        for result in raw_validation_results
    ),
)

print(
    "Valid article structure:",
    sum(
        result["is_valid_article"] is True
        for result in raw_validation_results
    ),
)

print(
    "Invalid article structure:",
    sum(
        result["is_valid_article"] is False
        for result in raw_validation_results
    ),
)

Total manifest records: 529
HTML files present: 529
Source files present: 529
Valid hashes: 529
Valid article structure: 527
Invalid article structure: 0


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## پیدا کردن دو رکورد بدون وضعیت Validation

دو مقاله فایل خام سالم و Hash معتبر دارند، اما فیلد `is_valid_article` در `source.json` آن‌ها ثبت نشده است.

احتمالاً این فایل‌ها قبل از اضافه شدن مرحله `Raw Validation` ایجاد شده‌اند.

در این مرحله فقط شناسه این دو مقاله را پیدا می‌کنیم.

</div>

In [48]:
# Code Cell 49 - Find records without validation status

missing_validation = [
    result
    for result in raw_validation_results
    if result["is_valid_article"] is None
]

print("Missing validation records:", len(missing_validation))

for result in missing_validation:
    print(result["article_id"])

Missing validation records: 2
00D061A8
00D9A87F


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تکمیل Validation دو مقاله قدیمی

دو مقاله `00D061A8` و `00D9A87F` فایل خام سالم و Hash معتبر دارند، اما چون قبل از اضافه‌شدن Validation دانلود شده‌اند، فیلدهای ساختاری در `source.json` آن‌ها وجود ندارد.

در این مرحله بدون دانلود مجدد:

- HTML موجود خوانده می‌شود.
- ساختار مقاله بررسی می‌شود.
- `has_h1`
- `has_content_container`
- `is_valid_article`

به `source.json` اضافه می‌شوند.

</div>

In [49]:
# Code Cell 50 - Update missing validation metadata

for article_id in [
    "00D061A8",
    "00D9A87F",
]:
    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    raw_bytes = html_path.read_bytes()

    structure = has_article_structure(
        raw_bytes
    )

    is_valid_article = (
        structure["has_h1"]
        and structure["has_content_container"]
    )

    with source_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        source_data = json.load(file)

    source_data["has_h1"] = structure["has_h1"]
    source_data["has_content_container"] = (
        structure["has_content_container"]
    )
    source_data["is_valid_article"] = (
        is_valid_article
    )

    with source_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            source_data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        article_id,
        "->",
        is_valid_article,
    )

00D061A8 -> True
00D9A87F -> True


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اعتبارسنجی نهایی کامل `Raw Snapshot`

حالا یک بار دیگر کل `529` مقاله بررسی می‌شوند تا مطمئن شویم:

- همه `article.html`ها وجود دارند.
- همه `source.json`ها وجود دارند.
- همه Hashها معتبرند.
- همه رکوردها `is_valid_article = True` دارند.
- هیچ رکورد بدون Validation باقی نمانده است.

</div>

In [50]:
# Code Cell 51 - Final raw snapshot validation

final_raw_validation = []

for record in website_manifest:
    article_id = record["article_id"]

    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    html_exists = html_path.exists()
    source_exists = source_path.exists()

    hash_valid = False
    is_valid_article = None

    if html_exists and source_exists:
        raw_bytes = html_path.read_bytes()

        with source_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            source_data = json.load(file)

        actual_hash = hashlib.sha256(
            raw_bytes
        ).hexdigest()

        hash_valid = (
            actual_hash
            == source_data.get("raw_content_hash")
        )

        is_valid_article = source_data.get(
            "is_valid_article"
        )

    final_raw_validation.append({
        "article_id": article_id,
        "html_exists": html_exists,
        "source_exists": source_exists,
        "hash_valid": hash_valid,
        "is_valid_article": is_valid_article,
    })


print("Total:", len(final_raw_validation))
print(
    "HTML present:",
    sum(x["html_exists"] for x in final_raw_validation),
)
print(
    "Source present:",
    sum(x["source_exists"] for x in final_raw_validation),
)
print(
    "Valid hashes:",
    sum(x["hash_valid"] for x in final_raw_validation),
)
print(
    "Valid articles:",
    sum(x["is_valid_article"] is True for x in final_raw_validation),
)
print(
    "Invalid articles:",
    sum(x["is_valid_article"] is False for x in final_raw_validation),
)
print(
    "Missing validation:",
    sum(x["is_valid_article"] is None for x in final_raw_validation),
)

Total: 529
HTML present: 529
Source present: 529
Valid hashes: 529
Valid articles: 529
Invalid articles: 0
Missing validation: 0


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## پایان مرحله `Raw Crawl`

مرحله `Raw Crawl` سایت با موفقیت کامل شد.

در این مرحله:

- تمام `529` مقاله کشف‌شده دانلود شدند.
- فایل خام هر مقاله ذخیره شد.
- `source.json` برای همه مقالات ثبت شد.
- `SHA-256` تمام فایل‌ها اعتبارسنجی شد.
- تمام `529` مقاله ساختار معتبر مقاله دارند.
- قابلیت‌های `Retry`، `Backoff`، `Resume` و `Error Handling` نیز تست شدند.

مرحله بعدی وارد `Extraction V2` می‌شود؛ یعنی استخراج متن و Metadata واقعی مقاله از همین Raw Snapshot.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## شروع `Extraction V2`

حالا وارد مرحله استخراج می‌شویم.

در قدم اول هنوز چیزی را روی هر `529` مقاله اجرا نمی‌کنیم؛ فقط یک مقاله نمونه را از `Raw HTML` می‌خوانیم و بخش‌های اصلی آن را بررسی می‌کنیم تا Selectorهای مناسب Extraction را قطعی کنیم.

مواردی که فعلاً بررسی می‌کنیم:

- عنوان مقاله
- بدنه مقاله
- نویسنده
- تاریخ
- دسته‌بندی یا Metadataهای قابل استخراج

</div>

In [51]:
# Code Cell 52 - Inspect one raw article for extraction

sample_article_id = website_manifest[0]["article_id"]

sample_html_path = (
    (DATA_ROOT / "phase2/raw/website")
    / sample_article_id
    / "article.html"
)

sample_raw_html = sample_html_path.read_bytes()

sample_soup = BeautifulSoup(
    sample_raw_html,
    "html.parser",
)

print("Article ID:", sample_article_id)
print()

print("Page title:")
print(
    sample_soup.title.get_text(
        " ",
        strip=True,
    ) if sample_soup.title else None
)

print()
print("H1:")
print(
    sample_soup.find("h1").get_text(
        " ",
        strip=True,
    ) if sample_soup.find("h1") else None
)

print()
print("Content container exists:")
print(
    sample_soup.select_one(
        ".blog-post-body-container .mt-md"
    ) is not None
)

Article ID: 008E61CC

Page title:
علف دادم شیر گرفتم

H1:
علف دادم شیر گرفتم

Content container exists:
True


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی متن استخراج‌شده مقاله نمونه

حالا باید ببینیم Selector فعلی بدنه واقعاً متن مفید مقاله را برمی‌گرداند یا نه.

در این مرحله فقط روی همان مقاله نمونه:

- متن بدنه استخراج می‌شود.
- طول متن اندازه‌گیری می‌شود.
- ابتدای متن نمایش داده می‌شود.

هدف این است که قبل از ساخت تابع نهایی Extraction مطمئن شویم محتوای واقعی مقاله استخراج می‌شود.

</div>

In [52]:
# Code Cell 53 - Inspect extracted body text

content_container = sample_soup.select_one(
    ".blog-post-body-container .mt-md"
)

sample_body_text = content_container.get_text(
    "\n",
    strip=True,
)

print("Extracted text length:", len(sample_body_text))
print()
print(sample_body_text[:2000])

Extracted text length: 1336

دویدم و دویدم،
سر کوهی رسیدم،
دو تا خاتونُ دیدم ...
این شعر از معروف­‌ترین شعرهای دوران کودکی ماست. ماجرای پسر بچه­‌ای که در مسیری طولانی با افرادی گوناگون مواجه می­‌شود و از هر کس چیزی می‌­گیرد و چیزی به آ‌ن­‌ها می‌دهد. چیزهایی که خودش احتیاجی به آن­‌ها ندارد، ولی نیازِ طرف مقابل را برطرف می­‌کند. احتمالاً آن زمان که سعی می‌­کردیم با تندترین ریتم ممکن این شعر را بخوانیم و در مسابقه برنده شویم، حواس‌مان نبود در داستان شعر، چه اتفاقی در حال رخ دادن است و احتمالاً کسی هم برای‌مان توضیح نداد به این حالت از داد و ستد، مبادله پایاپای یا کالا به کالا می­‌گویند. همان مبادله‌­ای که بشر برای اولین بار، برای رفع نیازهای خود و خلاصی از دست چیزهایی که دیگر احتیاجی به آن­‌ها نداشت، آن را اختراع کرد و عملاً علم اقتصاد را پایه‌­ریزی کرد.
با گذر زمان، نسخه­‌های متنوعی از این شعر با شخصیت­‌ها و کالاهای گوناگون سروده شده یا به داستان تبدیل شده است. اما مفهوم همه این شعرها و داستان­‌ها هنوز همان مبادله پایاپای است. کتاب «علف دادم، شیر گرفتم»، از مجموعه کتاب­‌های آموزش سواد ما

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی Metadata مقاله نمونه

حالا باید از همین HTML نمونه، اطلاعات جانبی مقاله را پیدا کنیم.

در این مرحله فقط ساختار صفحه را برای موارد زیر بررسی می‌کنیم:

- نویسنده
- تاریخ انتشار
- دسته‌بندی
- تگ‌ها

هنوز چیزی را روی کل ۵۲۹ مقاله اجرا نمی‌کنیم.

</div>

In [53]:
# Code Cell 54 - Inspect article metadata candidates

candidate_texts = []

for tag in sample_soup.find_all(
    ["a", "span", "div", "p", "time"]
):
    text = tag.get_text(
        " ",
        strip=True,
    )

    if not text:
        continue

    if any(
        keyword in text
        for keyword in [
            "نویسنده",
            "تاریخ",
            "دسته",
            "برچسب",
        ]
    ):
        candidate_texts.append(text)


for text in candidate_texts[:50]:
    print(text)

× فروشگاه انتشارات بر اساس موضوع‌ها خرج و پس‌انداز اعتبار و بدهی کاریابی و درآمد سرمایه‌گذاری ریسک و بیمه تصمیم‌گیری مالی سنجش سواد مالی آموزش سواد مالی عمومی بر اساس مخاطب کودکان نوجوانان جوانان بزرگسالان والدین فعالان بر اساس قالب ببینیم بشنویم بخوانیم بازی کنیم بر اساس هشتگ محصول آکادمی سواد مالی در جهان مربیان بخوانند سیاست‌گذاران بخوانند والدین بخوانند سواد مالی خانم‌ها سواد مالی در ادبیات برنامه رادیویی چهلستون گفتگوهای اختصاصی دوره های آموزشی بر اساس موضوع‌ها خرج و پس‌انداز اعتبار و بدهی کاریابی و درآمد سرمایه‌گذاری ریسک و بیمه تصمیم‌گیری مالی سنجش سواد مالی آموزش سواد مالی عمومی بر اساس مخاطب کودکان نوجوانان جوانان بزرگسالان والدین فعالان بر اساس قالب ببینیم بشنویم بخوانیم بازی کنیم بر اساس هشتگ محصول آکادمی سواد مالی در جهان مربیان بخوانند سیاست‌گذاران بخوانند والدین بخوانند سواد مالی خانم‌ها سواد مالی در ادبیات برنامه رادیویی چهلستون گفتگوهای اختصاصی مقالات بر اساس موضوع‌ها خرج و پس‌انداز اعتبار و بدهی کاریابی و درآمد سرمایه‌گذاری ریسک و بیمه تصمیم‌گیری مالی سنجش سواد مالی آم

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## پیدا کردن ساختار دقیق نویسنده

در این مرحله تگی که متن `نویسنده:` را دارد پیدا می‌کنیم و ساختار HTML اطراف آن را نمایش می‌دهیم.

هدف این است که به یک Selector پایدار برای استخراج `author` برسیم، نه اینکه نویسنده را با جست‌وجوی متنی کلی استخراج کنیم.

</div>

In [54]:
# Code Cell 55 - Inspect author HTML structure

author_text_node = sample_soup.find(
    string=lambda text: (
        text
        and "نویسنده:" in text
    )
)

if author_text_node:
    parent = author_text_node.parent

    print("Tag:", parent.name)
    print("Classes:", parent.get("class"))
    print("Text:", parent.get_text(" ", strip=True))
    print()
    print("Parent HTML:")
    print(parent.parent.prettify()[:3000])
else:
    print("Author text not found.")

Tag: span
Classes: ['post-author-name']
Text: نویسنده: سودابه فرخنده

Parent HTML:
<a class="post-author" href="/Author/Detail/6/%D8%B3%D9%88%D8%AF%D8%A7%D8%A8%D9%87-%D9%81%D8%B1%D8%AE%D9%86%D8%AF%D9%87" title="بازدید">
 <img alt="" class="post-author-image img-thumbnail img-thumbnail-hover-icon" src="https://cdn.fintelligence.ir/Uploaded/Author/Thumb/014130e7-59d0-46a1-965a-98deabb0995c.jpg"/>
 <span class="post-author-name">
  نویسنده: سودابه فرخنده
 </span>
</a>



<div dir="rtl" style="text-align: right; line-height: 1.8;">

## استخراج نویسنده مقاله نمونه

حالا نویسنده را با Selector دقیق استخراج می‌کنیم و پیشوند `نویسنده:` را حذف می‌کنیم.

همچنین `author_id` را از لینک نویسنده می‌گیریم تا Metadata کامل‌تری داشته باشیم.

</div>

In [55]:
# Code Cell 56 - Extract sample author metadata

author_link = sample_soup.select_one(
    "a.post-author"
)

author_name_tag = sample_soup.select_one(
    ".post-author-name"
)

author_name = None
author_id = None

if author_name_tag:
    author_name = author_name_tag.get_text(
        " ",
        strip=True,
    ).replace(
        "نویسنده:",
        "",
    ).strip()

if author_link:
    author_href = author_link.get(
        "href"
    )

    if author_href:
        parts = author_href.strip(
            "/"
        ).split("/")

        if (
            len(parts) >= 3
            and parts[0] == "Author"
            and parts[1] == "Detail"
        ):
            author_id = parts[2]


print("Author name:", author_name)
print("Author ID:", author_id)

Author name: سودابه فرخنده
Author ID: 6


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی تاریخ و Metadata مقاله

حالا باید ساختار HTML اطراف اطلاعاتی مثل تاریخ، زمان مطالعه و سایر Metadataهای مقاله را پیدا کنیم.

در این مرحله هنوز فقط همان مقاله نمونه بررسی می‌شود تا Selectorهای دقیق را مشخص کنیم.

</div>

In [56]:
# Code Cell 57 - Inspect article meta HTML

author_block = sample_soup.select_one(
    "a.post-author"
)

if author_block:
    current = author_block.parent

    print(
        current.prettify()[:5000]
    )
else:
    print("Author block not found.")

<div class="post-statistics mt-xs">
 <a class="post-author" href="/Author/Detail/6/%D8%B3%D9%88%D8%AF%D8%A7%D8%A8%D9%87-%D9%81%D8%B1%D8%AE%D9%86%D8%AF%D9%87" title="بازدید">
  <img alt="" class="post-author-image img-thumbnail img-thumbnail-hover-icon" src="https://cdn.fintelligence.ir/Uploaded/Author/Thumb/014130e7-59d0-46a1-965a-98deabb0995c.jpg"/>
  <span class="post-author-name">
   نویسنده: سودابه فرخنده
  </span>
 </a>
 <span class="post-visit" href="" title="بازدید">
  <i class="fa fa-eye">
  </i>
  4672
 </span>
 <span class="post-review hidden" data-id="" title="نظرات">
  <i class="fa fa-comment-o">
  </i>
  0
 </span>
 <a class="post-like hidden" href="" title="لایک">
  <i class="fa fa-heart-o">
  </i>
  0
 </a>
 <span class="post-studytime" href="" title="زمان مطالعه">
  <i class="fa fa-hourglass-half">
  </i>
  2
  <span>
   دقیقه
  </span>
 </span>
 <div style="clear: both">
 </div>
</div>



<div dir="rtl" style="text-align: right; line-height: 1.8;">

## پیدا کردن تاریخ انتشار مقاله

در این مرحله HTML مقاله نمونه برای عناصر احتمالی تاریخ بررسی می‌شود.

به‌جای جست‌وجوی متن کامل صفحه، کلاس‌ها و تگ‌هایی را بررسی می‌کنیم که نام آن‌ها شامل مواردی مثل:

- `date`
- `time`
- `publish`
- `created`

باشد.

</div>

In [57]:
# Code Cell 58 - Inspect date metadata candidates

date_candidates = []

for tag in sample_soup.find_all(True):
    classes = tag.get("class", [])

    class_text = " ".join(classes).lower()

    if any(
        keyword in class_text
        for keyword in [
            "date",
            "time",
            "publish",
            "created",
        ]
    ):
        text = tag.get_text(
            " ",
            strip=True,
        )

        if text:
            date_candidates.append({
                "tag": tag.name,
                "class": classes,
                "text": text,
            })


for item in date_candidates[:30]:
    print(item)

{'tag': 'span', 'class': ['post-studytime'], 'text': '2 دقیقه'}
{'tag': 'span', 'class': ['blog-post-list-item-studytime'], 'text': '3 دقیقه'}
{'tag': 'span', 'class': ['blog-post-list-item-studytime'], 'text': '4 دقیقه'}
{'tag': 'span', 'class': ['blog-post-list-item-studytime'], 'text': '3 دقیقه'}
{'tag': 'span', 'class': ['blog-post-list-item-studytime'], 'text': '2 دقیقه'}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی تاریخ در Metadata و Attributeهای صفحه

چون تاریخ انتشار در کلاس‌های HTML پیدا نشد، حالا این منابع بررسی می‌شوند:

- تگ‌های `meta`
- attribute به نام `datetime`
- attributeهای `data-*`
- داده‌های ساختاریافته `JSON-LD`

هدف این است که بدون حدس‌زدن، منبع واقعی تاریخ انتشار را در HTML پیدا کنیم.

</div>

In [58]:
# Code Cell 59 - Inspect structured date metadata

print("Meta candidates:")
for tag in sample_soup.find_all("meta"):
    name = (
        tag.get("name")
        or tag.get("property")
        or ""
    ).lower()

    content = tag.get("content")

    if any(
        keyword in name
        for keyword in [
            "date",
            "publish",
            "article",
            "created",
            "modified",
        ]
    ):
        print(
            name,
            "->",
            content,
        )


print()
print("Datetime candidates:")

for tag in sample_soup.find_all(
    attrs={"datetime": True}
):
    print(
        tag.name,
        "->",
        tag.get("datetime"),
        "|",
        tag.get_text(
            " ",
            strip=True,
        ),
    )


print()
print("JSON-LD candidates:")

for script in sample_soup.find_all(
    "script",
    type="application/ld+json",
):
    text = script.get_text(
        strip=True,
    )

    print(text[:3000])

Meta candidates:

Datetime candidates:

JSON-LD candidates:


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## استخراج Metadataهای قابل اتکا

برای مقاله نمونه، تاریخ انتشار در HTML خام پیدا نشد.

بنابراین در `Extraction V2` فعلاً فقط Metadataهایی استخراج می‌شوند که منبع مستقیم و قابل‌اعتماد داخل صفحه دارند.

در این مرحله:

- `view_count`
- `study_time_minutes`

را از ساختار مقاله استخراج می‌کنیم.

</div>

In [59]:
# Code Cell 60 - Extract sample article statistics

visit_tag = sample_soup.select_one(
    ".post-visit"
)

study_time_tag = sample_soup.select_one(
    ".post-studytime"
)

view_count = None
study_time_minutes = None

if visit_tag:
    visit_text = visit_tag.get_text(
        " ",
        strip=True,
    )

    digits = "".join(
        char
        for char in visit_text
        if char.isdigit()
    )

    if digits:
        view_count = int(digits)


if study_time_tag:
    study_text = study_time_tag.get_text(
        " ",
        strip=True,
    )

    digits = "".join(
        char
        for char in study_text
        if char.isdigit()
    )

    if digits:
        study_time_minutes = int(digits)


print("View count:", view_count)
print("Study time minutes:", study_time_minutes)

View count: 4672
Study time minutes: 2


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی هشتگ‌ها و برچسب‌های مقاله

حالا باید هشتگ‌های مرتبط با خود مقاله را پیدا کنیم.

برای مقاله نمونه در صفحه عبارت `هشتگ‌های مرتبط` وجود دارد، اما باید ساختار HTML دقیق اطراف آن را ببینیم تا فقط Tagهای همان مقاله استخراج شوند و منوها یا مطالب پیشنهادی وارد داده نشوند.

</div>

In [60]:
# Code Cell 61 - Inspect article tags structure

tags_text_node = sample_soup.find(
    string=lambda text: (
        text
        and "هشتگ‌های مرتبط" in text
    )
)

if tags_text_node:
    parent = tags_text_node.parent

    print("Tag:", parent.name)
    print("Classes:", parent.get("class"))
    print()
    print(parent.parent.prettify()[:5000])
else:
    print("Related tags section not found.")

Tag: h4
Classes: ['color-yellow-gold', 'mt-md']

<div class="bg-white-border-radius-10 detail-container">
 <div class="blog-post-body-container">
  <div class="mt-md">
   <h2 class="paragraph-title-1">
   </h2>
   <div class="row">
    <div class="col-sm-12">
     <p class="text-justify">
      <p style="text-align: justify;">
       دویدم و دویدم،
      </p>
      <p style="text-align: justify;">
       سر کوهی رسیدم،
      </p>
      <p style="text-align: justify;">
       دو تا خاتونُ دیدم ...
      </p>
      <p style="text-align: justify;">
       این شعر از معروف­‌ترین شعرهای دوران کودکی ماست. ماجرای پسر بچه­‌ای که در مسیری طولانی با افرادی گوناگون مواجه می­‌شود و از هر کس چیزی می‌­گیرد و چیزی به آ‌ن­‌ها می‌دهد. چیزهایی که خودش احتیاجی به آن­‌ها ندارد، ولی نیازِ طرف مقابل را برطرف می­‌کند. احتمالاً آن زمان که سعی می‌­کردیم با تندترین ریتم ممکن این شعر را بخوانیم و در مسابقه برنده شویم، حواس‌مان نبود در داستان شعر، چه اتفاقی در حال رخ دادن است و احتمالاً کسی هم برای‌مان توضیح نداد

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## استخراج Tagهای مقاله نمونه

در این مرحله برای هر Tag دو مقدار استخراج می‌کنیم:

- نام Tag
- `tag_id`

این اطلاعات بعداً می‌تواند برای Metadata، فیلتر و Retrieval مفید باشد.

</div>

In [61]:
# Code Cell 62 - Extract sample article tags

sample_tags = []

for tag_link in sample_soup.select(
    ".post-tag-list a"
):
    tag_name = tag_link.get_text(
        " ",
        strip=True,
    )

    tag_href = tag_link.get(
        "href"
    )

    tag_id = None

    if tag_href:
        parts = tag_href.strip(
            "/"
        ).split("/")

        if (
            len(parts) >= 3
            and parts[0] == "BlogTag"
            and parts[1] == "Detail"
        ):
            tag_id = parts[2]

    sample_tags.append({
        "tag_id": tag_id,
        "name": tag_name,
    })


print("Tags:", len(sample_tags))

for tag in sample_tags:
    print(tag)

Tags: 6
{'tag_id': '22', 'name': 'مبادله پایاپای'}
{'tag_id': '25', 'name': 'دو طرف معامله'}
{'tag_id': '54', 'name': 'محصول آکادمی'}
{'tag_id': '1028', 'name': 'مربیان بخوانند'}
{'tag_id': '1031', 'name': 'والدین بخوانند'}
{'tag_id': '2059', 'name': 'معرفی کتاب\u200cهای آکادمی'}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ساخت تابع اولیه `Extraction V2`

حالا اطلاعاتی که روی مقاله نمونه تأیید کرده‌ایم داخل یک تابع واحد جمع می‌شوند.

این نسخه فعلاً استخراج می‌کند:

- `title`
- `body_text`
- `author`
- `author_id`
- `view_count`
- `study_time_minutes`
- `tags`

هنوز تابع روی کل `529` مقاله اجرا نمی‌شود؛ ابتدا فقط روی همان مقاله نمونه تست می‌کنیم.

</div>

In [62]:
# Code Cell 63 - Build initial extraction V2 function

def extract_article_v2(raw_bytes):
    soup = BeautifulSoup(
        raw_bytes,
        "html.parser",
    )

    title_tag = soup.find("h1")

    title = (
        title_tag.get_text(
            " ",
            strip=True,
        )
        if title_tag
        else None
    )

    content_container = soup.select_one(
        ".blog-post-body-container .mt-md"
    )

    body_text = (
        content_container.get_text(
            "\n",
            strip=True,
        )
        if content_container
        else None
    )

    author_name = None
    author_id = None

    author_name_tag = soup.select_one(
        ".post-author-name"
    )

    author_link = soup.select_one(
        "a.post-author"
    )

    if author_name_tag:
        author_name = author_name_tag.get_text(
            " ",
            strip=True,
        ).replace(
            "نویسنده:",
            "",
        ).strip()

    if author_link:
        author_href = author_link.get(
            "href"
        )

        if author_href:
            parts = author_href.strip(
                "/"
            ).split("/")

            if (
                len(parts) >= 3
                and parts[0] == "Author"
                and parts[1] == "Detail"
            ):
                author_id = parts[2]

    view_count = None

    visit_tag = soup.select_one(
        ".post-visit"
    )

    if visit_tag:
        visit_digits = "".join(
            char
            for char in visit_tag.get_text(
                " ",
                strip=True,
            )
            if char.isdigit()
        )

        if visit_digits:
            view_count = int(
                visit_digits
            )

    study_time_minutes = None

    study_time_tag = soup.select_one(
        ".post-studytime"
    )

    if study_time_tag:
        study_digits = "".join(
            char
            for char in study_time_tag.get_text(
                " ",
                strip=True,
            )
            if char.isdigit()
        )

        if study_digits:
            study_time_minutes = int(
                study_digits
            )

    tags = []

    for tag_link in soup.select(
        ".post-tag-list a"
    ):
        tag_name = tag_link.get_text(
            " ",
            strip=True,
        )

        tag_href = tag_link.get(
            "href"
        )

        tag_id = None

        if tag_href:
            parts = tag_href.strip(
                "/"
            ).split("/")

            if (
                len(parts) >= 3
                and parts[0] == "BlogTag"
                and parts[1] == "Detail"
            ):
                tag_id = parts[2]

        tags.append({
            "tag_id": tag_id,
            "name": tag_name,
        })

    return {
        "title": title,
        "body_text": body_text,
        "author": author_name,
        "author_id": author_id,
        "view_count": view_count,
        "study_time_minutes": study_time_minutes,
        "tags": tags,
    }

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست تابع `Extraction V2`

در این مرحله تابع `extract_article_v2()` روی مقاله نمونه اجرا می‌شود.

هدف این است که قبل از اجرای گسترده روی کل `529` مقاله، مطمئن شویم همه فیلدهای اصلی درست برمی‌گردند.

</div>

In [63]:
# Code Cell 64 - Test extraction V2 on sample article

sample_extracted = extract_article_v2(
    sample_raw_html
)

print("Title:", sample_extracted["title"])
print("Author:", sample_extracted["author"])
print("Author ID:", sample_extracted["author_id"])
print("View count:", sample_extracted["view_count"])
print("Study time:", sample_extracted["study_time_minutes"])
print("Tags:", len(sample_extracted["tags"]))
print("Body length:", len(sample_extracted["body_text"] or ""))
print()

print(sample_extracted["body_text"][:1000])

Title: علف دادم شیر گرفتم
Author: سودابه فرخنده
Author ID: 6
View count: 4672
Study time: 2
Tags: 6
Body length: 1336

دویدم و دویدم،
سر کوهی رسیدم،
دو تا خاتونُ دیدم ...
این شعر از معروف­‌ترین شعرهای دوران کودکی ماست. ماجرای پسر بچه­‌ای که در مسیری طولانی با افرادی گوناگون مواجه می­‌شود و از هر کس چیزی می‌­گیرد و چیزی به آ‌ن­‌ها می‌دهد. چیزهایی که خودش احتیاجی به آن­‌ها ندارد، ولی نیازِ طرف مقابل را برطرف می­‌کند. احتمالاً آن زمان که سعی می‌­کردیم با تندترین ریتم ممکن این شعر را بخوانیم و در مسابقه برنده شویم، حواس‌مان نبود در داستان شعر، چه اتفاقی در حال رخ دادن است و احتمالاً کسی هم برای‌مان توضیح نداد به این حالت از داد و ستد، مبادله پایاپای یا کالا به کالا می­‌گویند. همان مبادله‌­ای که بشر برای اولین بار، برای رفع نیازهای خود و خلاصی از دست چیزهایی که دیگر احتیاجی به آن­‌ها نداشت، آن را اختراع کرد و عملاً علم اقتصاد را پایه‌­ریزی کرد.
با گذر زمان، نسخه­‌های متنوعی از این شعر با شخصیت­‌ها و کالاهای گوناگون سروده شده یا به داستان تبدیل شده است. اما مفهوم همه این شعرها و داستان­‌ها ه

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست `Extraction V2` روی چند مقاله

قبل از اجرای Extraction روی کل `529` مقاله، تابع را روی چند مقاله مختلف تست می‌کنیم.

هدف این مرحله بررسی این موارد است:

- خالی نبودن `title`
- خالی نبودن `body_text`
- وجود نویسنده
- رفتار درست `tags`
- مشاهده تفاوت طول متن بین مقالات

</div>

In [64]:
# Code Cell 65 - Test extraction V2 on a small batch

batch_extraction_results = []

for record in website_manifest[:5]:
    article_id = record["article_id"]

    html_path = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
        / "article.html"
    )

    raw_bytes = html_path.read_bytes()

    extracted = extract_article_v2(
        raw_bytes
    )

    result = {
        "article_id": article_id,
        "title": extracted["title"],
        "author": extracted["author"],
        "body_length": len(
            extracted["body_text"] or ""
        ),
        "tags_count": len(
            extracted["tags"]
        ),
    }

    batch_extraction_results.append(
        result
    )

    print(result)

{'article_id': '008E61CC', 'title': 'علف دادم شیر گرفتم', 'author': 'سودابه فرخنده', 'body_length': 1336, 'tags_count': 6}
{'article_id': '00D061A8', 'title': 'وقایع\u200cنگاری\u200cهای سنجش سواد مالی- 10', 'author': 'عاطفه جلادتی', 'body_length': 1684, 'tags_count': 3}
{'article_id': '00D9A87F', 'title': 'چطور شغل\u200cمان را توسعه دهیم؟', 'author': 'دکتر کمیل رودی', 'body_length': 0, 'tags_count': 4}
{'article_id': '00EDA828', 'title': 'وقایع\u200cنگاری\u200cهای سنجش سواد مالی-4', 'author': 'عاطفه جلادتی', 'body_length': 474, 'tags_count': 2}
{'article_id': '0152EE71', 'title': 'چرا شاغلان پاره\u200cوقت درآمدشان را بیشتر از واقع تخمین می\u200cزنند؟', 'author': 'حمیده معماریان', 'body_length': 1042, 'tags_count': 3}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی مقاله با بدنه خالی

برای مقاله `00D9A87F`، Selector فعلی:

`.blog-post-body-container .mt-md`

وجود دارد اما متن مفیدی برنمی‌گرداند.

در این مرحله ساختار HTML همین مقاله را بررسی می‌کنیم تا مشخص شود محتوای اصلی در چه Container دیگری قرار دارد.

</div>

In [65]:
# Code Cell 66 - Inspect article with empty extracted body

empty_article_id = "00D9A87F"

empty_html_path = (
    (DATA_ROOT / "phase2/raw/website")
    / empty_article_id
    / "article.html"
)

empty_raw_html = empty_html_path.read_bytes()

empty_soup = BeautifulSoup(
    empty_raw_html,
    "html.parser",
)

containers = empty_soup.select(
    ".blog-post-body-container"
)

print(
    "Body containers:",
    len(containers),
)

for index, container in enumerate(
    containers,
    start=1,
):
    text = container.get_text(
        " ",
        strip=True,
    )

    print(
        f"Container {index}:",
        "length =",
        len(text),
    )

    print(text[:1000])
    print("-" * 80)

Body containers: 2
Container 1: length = 0

--------------------------------------------------------------------------------
Container 2: length = 224
در شرایطی که شغل داریم و شغل‌مان را دوست داریم، برای توسعه شغل خود چه کنیم؟ مسیر امنیت شغلی چیست؟ درآمد غیرفعال چیست؟ شغل دوم چه ویژگی‌هایی دارد؟ سبک‌های شغلی پنج‌گانه کدامند؟ استخدام و کارآفرینی چه جایگاهی نسبت به هم دارند؟
--------------------------------------------------------------------------------


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اصلاح منطق استخراج بدنه مقاله

تست نشان داد که در بعضی مقالات، اولین `blog-post-body-container` خالی است و محتوای اصلی در Container بعدی قرار دارد.

بنابراین دیگر نباید فقط اولین Selector را انتخاب کنیم.

در این مرحله تابعی می‌سازیم که:

- همه `blog-post-body-container`ها را بررسی کند.
- Containerهای خالی را حذف کند.
- متن مناسب مقاله را برگرداند.

فعلاً فقط همین مقاله مشکل‌دار را تست می‌کنیم.

</div>

In [66]:
# Code Cell 67 - Build better body extraction

def extract_body_text_v2(soup):
    candidates = []

    for container in soup.select(
        ".blog-post-body-container"
    ):
        text = container.get_text(
            "\n",
            strip=True,
        )

        if text:
            candidates.append(text)

    if not candidates:
        return None

    return max(
        candidates,
        key=len,
    )


empty_body_v2 = extract_body_text_v2(
    empty_soup
)

print(
    "Extracted body length:",
    len(empty_body_v2 or ""),
)

print()
print(empty_body_v2[:1500])

Extracted body length: 224

در شرایطی که شغل داریم و شغل‌مان را دوست داریم، برای توسعه شغل خود چه کنیم؟
مسیر امنیت شغلی چیست؟
درآمد غیرفعال چیست؟
شغل دوم چه ویژگی‌هایی دارد؟
سبک‌های شغلی پنج‌گانه کدامند؟
استخدام و کارآفرینی چه جایگاهی نسبت به هم دارند؟


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اتصال منطق جدید بدنه به `Extraction V2`

حالا تابع اصلی `extract_article_v2()` را اصلاح می‌کنیم تا به‌جای استفاده از اولین Container، از تابع `extract_body_text_v2()` استفاده کند.

بقیه منطق استخراج Metadata تغییری نمی‌کند.

</div>

In [67]:
# Code Cell 68 - Update extraction V2 body logic

def extract_article_v2(raw_bytes):
    soup = BeautifulSoup(
        raw_bytes,
        "html.parser",
    )

    title_tag = soup.find("h1")

    title = (
        title_tag.get_text(
            " ",
            strip=True,
        )
        if title_tag
        else None
    )

    body_text = extract_body_text_v2(
        soup
    )

    author_name = None
    author_id = None

    author_name_tag = soup.select_one(
        ".post-author-name"
    )

    author_link = soup.select_one(
        "a.post-author"
    )

    if author_name_tag:
        author_name = author_name_tag.get_text(
            " ",
            strip=True,
        ).replace(
            "نویسنده:",
            "",
        ).strip()

    if author_link:
        author_href = author_link.get(
            "href"
        )

        if author_href:
            parts = author_href.strip(
                "/"
            ).split("/")

            if (
                len(parts) >= 3
                and parts[0] == "Author"
                and parts[1] == "Detail"
            ):
                author_id = parts[2]

    view_count = None

    visit_tag = soup.select_one(
        ".post-visit"
    )

    if visit_tag:
        visit_digits = "".join(
            char
            for char in visit_tag.get_text(
                " ",
                strip=True,
            )
            if char.isdigit()
        )

        if visit_digits:
            view_count = int(
                visit_digits
            )

    study_time_minutes = None

    study_time_tag = soup.select_one(
        ".post-studytime"
    )

    if study_time_tag:
        study_digits = "".join(
            char
            for char in study_time_tag.get_text(
                " ",
                strip=True,
            )
            if char.isdigit()
        )

        if study_digits:
            study_time_minutes = int(
                study_digits
            )

    tags = []

    for tag_link in soup.select(
        ".post-tag-list a"
    ):
        tag_name = tag_link.get_text(
            " ",
            strip=True,
        )

        tag_href = tag_link.get(
            "href"
        )

        tag_id = None

        if tag_href:
            parts = tag_href.strip(
                "/"
            ).split("/")

            if (
                len(parts) >= 3
                and parts[0] == "BlogTag"
                and parts[1] == "Detail"
            ):
                tag_id = parts[2]

        tags.append({
            "tag_id": tag_id,
            "name": tag_name,
        })

    return {
        "title": title,
        "body_text": body_text,
        "author": author_name,
        "author_id": author_id,
        "view_count": view_count,
        "study_time_minutes": study_time_minutes,
        "tags": tags,
    }

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست مجدد `Extraction V2` روی پنج مقاله

بعد از اصلاح منطق انتخاب بدنه، همان پنج مقاله قبلی دوباره بررسی می‌شوند.

هدف:

- اطمینان از رفع `body_length = 0`
- بررسی اینکه سایر مقالات همچنان درست استخراج می‌شوند
- مقایسه طول بدنه‌ها بعد از اصلاح

</div>

In [68]:
# Code Cell 69 - Retest extraction V2 on small batch

batch_extraction_results_v2 = []

for record in website_manifest[:5]:
    article_id = record["article_id"]

    html_path = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
        / "article.html"
    )

    raw_bytes = html_path.read_bytes()

    extracted = extract_article_v2(
        raw_bytes
    )

    result = {
        "article_id": article_id,
        "title": extracted["title"],
        "author": extracted["author"],
        "body_length": len(
            extracted["body_text"] or ""
        ),
        "tags_count": len(
            extracted["tags"]
        ),
    }

    batch_extraction_results_v2.append(
        result
    )

    print(result)

{'article_id': '008E61CC', 'title': 'علف دادم شیر گرفتم', 'author': 'سودابه فرخنده', 'body_length': 1336, 'tags_count': 6}
{'article_id': '00D061A8', 'title': 'وقایع\u200cنگاری\u200cهای سنجش سواد مالی- 10', 'author': 'عاطفه جلادتی', 'body_length': 1684, 'tags_count': 3}
{'article_id': '00D9A87F', 'title': 'چطور شغل\u200cمان را توسعه دهیم؟', 'author': 'دکتر کمیل رودی', 'body_length': 224, 'tags_count': 4}
{'article_id': '00EDA828', 'title': 'وقایع\u200cنگاری\u200cهای سنجش سواد مالی-4', 'author': 'عاطفه جلادتی', 'body_length': 2984, 'tags_count': 2}
{'article_id': '0152EE71', 'title': 'چرا شاغلان پاره\u200cوقت درآمدشان را بیشتر از واقع تخمین می\u200cزنند؟', 'author': 'حمیده معماریان', 'body_length': 1579, 'tags_count': 3}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## پیش‌اعتبارسنجی `Extraction V2` روی کل مجموعه

حالا تابع روی هر `529` مقاله اجرا می‌شود، اما هنوز هیچ خروجی نهایی ذخیره نمی‌کنیم.

هدف فقط بررسی کیفیت Extraction است:

- چند مقاله عنوان خالی دارند؟
- چند مقاله بدنه خالی دارند؟
- چند مقاله نویسنده ندارند؟
- کوتاه‌ترین بدنه‌ها کدام‌اند؟

اگر این تست سالم باشد، بعد وارد ذخیره خروجی Extraction می‌شویم.

</div>

In [69]:
# Code Cell 70 - Validate extraction V2 across all articles

extraction_validation = []

for record in website_manifest:
    article_id = record["article_id"]

    html_path = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
        / "article.html"
    )

    raw_bytes = html_path.read_bytes()

    extracted = extract_article_v2(
        raw_bytes
    )

    extraction_validation.append({
        "article_id": article_id,
        "title": extracted["title"],
        "author": extracted["author"],
        "body_length": len(
            extracted["body_text"] or ""
        ),
        "tags_count": len(
            extracted["tags"]
        ),
    })


print("Total:", len(extraction_validation))

print(
    "Missing title:",
    sum(
        not item["title"]
        for item in extraction_validation
    ),
)

print(
    "Missing body:",
    sum(
        item["body_length"] == 0
        for item in extraction_validation
    ),
)

print(
    "Missing author:",
    sum(
        not item["author"]
        for item in extraction_validation
    ),
)

print()
print("10 shortest bodies:")

for item in sorted(
    extraction_validation,
    key=lambda x: x["body_length"],
)[:10]:
    print(item)

Total: 529
Missing title: 0
Missing body: 0
Missing author: 0

10 shortest bodies:
{'article_id': '00D9A87F', 'title': 'چطور شغل\u200cمان را توسعه دهیم؟', 'author': 'دکتر کمیل رودی', 'body_length': 224, 'tags_count': 4}
{'article_id': '7E223060', 'title': 'ماجرای صندوق سهام دولتی', 'author': 'صاحب\u200cنظران', 'body_length': 227, 'tags_count': 5}
{'article_id': 'BF2CFA69', 'title': 'کار داوطلبانه، شروعی برای کاریابی و کسب درآمد', 'author': 'دکتر کمیل رودی', 'body_length': 262, 'tags_count': 2}
{'article_id': '2E656F0E', 'title': 'سه دلیل برای سرمایه\u200cگذاری و پنج گزینه برای انتخاب', 'author': 'دکتر کمیل رودی', 'body_length': 300, 'tags_count': 5}
{'article_id': 'C203F9BA', 'title': 'چرا ریسک\u200cپذیر نیستیم؟ با نگاهی به صندوق\u200cهای سرمایه\u200cگذاری', 'author': 'دکتر کمیل رودی', 'body_length': 310, 'tags_count': 1}
{'article_id': '18F4E47E', 'title': 'پادکست | فیه ما فی-20', 'author': 'هادی نجف', 'body_length': 440, 'tags_count': 1}
{'article_id': '2486A245', 'title': 'پادکست | 

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ساخت خروجی ساخت‌یافته `Extraction V2`

حالا برای هر مقاله یک رکورد استاندارد استخراج می‌کنیم که شامل این موارد است:

- `source_id`
- `article_id`
- `title`
- `body_text`
- `author`
- `author_id`
- `knowledge_owner`
- `content_origin`
- `view_count`
- `study_time_minutes`
- `tags`
- `canonical_url`
- `raw_content_hash`

خروجی فعلاً در حافظه ساخته می‌شود؛ هنوز فایل نهایی ذخیره نمی‌کنیم.

</div>

In [70]:
# Code Cell 71 - Build extracted website records

extracted_website_records = []

for record in website_manifest:
    article_id = record["article_id"]

    article_dir = (
        (DATA_ROOT / "phase2/raw/website")
        / article_id
    )

    html_path = article_dir / "article.html"
    source_path = article_dir / "source.json"

    raw_bytes = html_path.read_bytes()

    with source_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        source_data = json.load(file)

    extracted = extract_article_v2(
        raw_bytes
    )

    extracted_record = {
        "source_id": record["source_id"],
        "source_type": "website",
        "article_id": article_id,
        "title": extracted["title"],
        "body_text": extracted["body_text"],
        "author": extracted["author"],
        "author_id": extracted["author_id"],
        "knowledge_owner": record["knowledge_owner"],
        "content_origin": record["content_origin"],
        "view_count": extracted["view_count"],
        "study_time_minutes": extracted["study_time_minutes"],
        "tags": extracted["tags"],
        "url": record["url"],
        "canonical_url": record["canonical_url"],
        "raw_content_hash": source_data["raw_content_hash"],
    }

    extracted_website_records.append(
        extracted_record
    )


print(
    "Extracted records:",
    len(extracted_website_records),
)

print()
print(
    extracted_website_records[0]
)

Extracted records: 529

{'source_id': 'website:008E61CC', 'source_type': 'website', 'article_id': '008E61CC', 'title': 'علف دادم شیر گرفتم', 'body_text': 'دویدم و دویدم،\nسر کوهی رسیدم،\nدو تا خاتونُ دیدم ...\nاین شعر از معروف\xad\u200cترین شعرهای دوران کودکی ماست. ماجرای پسر بچه\xad\u200cای که در مسیری طولانی با افرادی گوناگون مواجه می\xad\u200cشود و از هر کس چیزی می\u200c\xadگیرد و چیزی به آ\u200cن\xad\u200cها می\u200cدهد. چیزهایی که خودش احتیاجی به آن\xad\u200cها ندارد، ولی نیازِ طرف مقابل را برطرف می\xad\u200cکند. احتمالاً آن زمان که سعی می\u200c\xadکردیم با تندترین ریتم ممکن این شعر را بخوانیم و در مسابقه برنده شویم، حواس\u200cمان نبود در داستان شعر، چه اتفاقی در حال رخ دادن است و احتمالاً کسی هم برای\u200cمان توضیح نداد به این حالت از داد و ستد، مبادله پایاپای یا کالا به کالا می\xad\u200cگویند. همان مبادله\u200c\xadای که بشر برای اولین بار، برای رفع نیازهای خود و خلاصی از دست چیزهایی که دیگر احتیاجی به آن\xad\u200cها نداشت، آن را اختراع کرد و عملاً علم اقتصاد را پایه\u200c\xadریز

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## نرمال‌سازی متن استخراج‌شده

در متن خام بعضی مقالات کاراکترهای نامرئی مانند `Soft Hyphen` وجود دارد.

این کاراکترها ممکن است در مراحل بعدی باعث ایجاد تفاوت‌های مصنوعی در:

- Chunking
- Embedding
- Search
- Deduplication

شوند.

در این مرحله فقط یک تابع ساده برای پاک‌سازی این نویزهای متنی می‌سازیم.

</div>

In [71]:
# Code Cell 72 - Build text normalization function

import re
import unicodedata


def normalize_extracted_text(text):
    if text is None:
        return None

    text = unicodedata.normalize(
        "NFC",
        text,
    )

    text = text.replace(
        "\u00ad",
        "",
    )

    text = text.replace(
        "\ufeff",
        "",
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text,
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## تست نرمال‌سازی متن

در این مرحله متن مقاله نمونه قبل و بعد از `normalize_extracted_text()` مقایسه می‌شود.

هدف این است که مطمئن شویم کاراکترهای نامرئی حذف شده‌اند، بدون اینکه متن اصلی خراب شود.

</div>

In [72]:
# Code Cell 73 - Test text normalization

sample_body_before = (
    extracted_website_records[0]["body_text"]
)

sample_body_after = normalize_extracted_text(
    sample_body_before
)

print(
    "Before length:",
    len(sample_body_before),
)

print(
    "After length:",
    len(sample_body_after),
)

print(
    "Soft hyphen before:",
    sample_body_before.count("\u00ad"),
)

print(
    "Soft hyphen after:",
    sample_body_after.count("\u00ad"),
)

print()
print(sample_body_after[:1000])

Before length: 1336
After length: 1316
Soft hyphen before: 20
Soft hyphen after: 0

دویدم و دویدم،
سر کوهی رسیدم،
دو تا خاتونُ دیدم ...
این شعر از معروف‌ترین شعرهای دوران کودکی ماست. ماجرای پسر بچه‌ای که در مسیری طولانی با افرادی گوناگون مواجه می‌شود و از هر کس چیزی می‌گیرد و چیزی به آ‌ن‌ها می‌دهد. چیزهایی که خودش احتیاجی به آن‌ها ندارد، ولی نیازِ طرف مقابل را برطرف می‌کند. احتمالاً آن زمان که سعی می‌کردیم با تندترین ریتم ممکن این شعر را بخوانیم و در مسابقه برنده شویم، حواس‌مان نبود در داستان شعر، چه اتفاقی در حال رخ دادن است و احتمالاً کسی هم برای‌مان توضیح نداد به این حالت از داد و ستد، مبادله پایاپای یا کالا به کالا می‌گویند. همان مبادله‌ای که بشر برای اولین بار، برای رفع نیازهای خود و خلاصی از دست چیزهایی که دیگر احتیاجی به آن‌ها نداشت، آن را اختراع کرد و عملاً علم اقتصاد را پایه‌ریزی کرد.
با گذر زمان، نسخه‌های متنوعی از این شعر با شخصیت‌ها و کالاهای گوناگون سروده شده یا به داستان تبدیل شده است. اما مفهوم همه این شعرها و داستان‌ها هنوز همان مبادله پایاپای است. کتاب «علف دادم، شیر گ

<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اعمال نرمال‌سازی روی رکوردهای استخراج‌شده

حالا نرمال‌سازی را روی فیلدهای متنی همه `529` رکورد اعمال می‌کنیم.

در این مرحله این فیلدها پاک‌سازی می‌شوند:

- `title`
- `body_text`
- `author`
- نام Tagها

هنوز فایل نهایی ذخیره نمی‌کنیم؛ ابتدا نتیجه را در حافظه اعتبارسنجی می‌کنیم.

</div>

In [73]:
# Code Cell 74 - Normalize all extracted records

for record in extracted_website_records:
    record["title"] = normalize_extracted_text(
        record["title"]
    )

    record["body_text"] = normalize_extracted_text(
        record["body_text"]
    )

    record["author"] = normalize_extracted_text(
        record["author"]
    )

    for tag in record["tags"]:
        tag["name"] = normalize_extracted_text(
            tag["name"]
        )


print(
    "Normalized records:",
    len(extracted_website_records),
)

print(
    "Remaining soft hyphens:",
    sum(
        (record["body_text"] or "").count("\u00ad")
        for record in extracted_website_records
    ),
)

print()
print(
    extracted_website_records[0]["body_text"][:500]
)

Normalized records: 529
Remaining soft hyphens: 0

دویدم و دویدم،
سر کوهی رسیدم،
دو تا خاتونُ دیدم ...
این شعر از معروف‌ترین شعرهای دوران کودکی ماست. ماجرای پسر بچه‌ای که در مسیری طولانی با افرادی گوناگون مواجه می‌شود و از هر کس چیزی می‌گیرد و چیزی به آ‌ن‌ها می‌دهد. چیزهایی که خودش احتیاجی به آن‌ها ندارد، ولی نیازِ طرف مقابل را برطرف می‌کند. احتمالاً آن زمان که سعی می‌کردیم با تندترین ریتم ممکن این شعر را بخوانیم و در مسابقه برنده شویم، حواس‌مان نبود در داستان شعر، چه اتفاقی در حال رخ دادن است و احتمالاً کسی هم برای‌مان توضیح نداد به این حالت از


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## اعتبارسنجی نهایی رکوردهای Extract شده

قبل از ذخیره فایل نهایی، یک Validation نهایی انجام می‌دهیم تا مطمئن شویم:

- تعداد رکوردها همان `529` است.
- `source_id` تکراری نداریم.
- `article_id` تکراری نداریم.
- عنوان خالی نداریم.
- متن خالی نداریم.
- نویسنده خالی نداریم.
- `raw_content_hash` خالی نداریم.

</div>

In [74]:
# Code Cell 75 - Final validation of extracted records

print(
    "Total records:",
    len(extracted_website_records),
)

print(
    "Unique source IDs:",
    len({
        record["source_id"]
        for record in extracted_website_records
    }),
)

print(
    "Unique article IDs:",
    len({
        record["article_id"]
        for record in extracted_website_records
    }),
)

print(
    "Missing titles:",
    sum(
        not record["title"]
        for record in extracted_website_records
    ),
)

print(
    "Missing bodies:",
    sum(
        not record["body_text"]
        for record in extracted_website_records
    ),
)

print(
    "Missing authors:",
    sum(
        not record["author"]
        for record in extracted_website_records
    ),
)

print(
    "Missing raw hashes:",
    sum(
        not record["raw_content_hash"]
        for record in extracted_website_records
    ),
)

Total records: 529
Unique source IDs: 529
Unique article IDs: 529
Missing titles: 0
Missing bodies: 0
Missing authors: 0
Missing raw hashes: 0


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ذخیره خروجی `Extraction V2`

در این مرحله رکوردهای نهایی استخراج‌شده در مسیر `interim` ذخیره می‌شوند.

این فایل هنوز Dataset نهایی برای Chunking نیست؛ بلکه خروجی تمیز و ساخت‌یافته مرحله Extraction است.

</div>

In [75]:
# Code Cell 76 - Save extracted website records

extracted_output_path = (DATA_ROOT / "phase2/interim/website_extracted_v2.json")

extracted_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with extracted_output_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        extracted_website_records,
        file,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Saved:",
    extracted_output_path,
)

print(
    "Records:",
    len(extracted_website_records),
)

print(
    "Size bytes:",
    extracted_output_path.stat().st_size,
)

Saved: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/phase2/interim/website_extracted_v2.json
Records: 529
Size bytes: 2263917


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی فایل ذخیره‌شده

حالا فایل را دوباره از دیسک می‌خوانیم تا مطمئن شویم:

- فایل سالم است.
- تعداد رکوردها همان `529` است.
- داده Reload شده دقیقاً با داده داخل حافظه برابر است.

</div>

In [76]:
# Code Cell 77 - Reload and verify extracted website file

with extracted_output_path.open(
    "r",
    encoding="utf-8",
) as file:
    reloaded_extracted_records = json.load(file)


print(
    "Reloaded records:",
    len(reloaded_extracted_records),
)

print(
    "Matches in-memory data:",
    reloaded_extracted_records
    == extracted_website_records,
)

Reloaded records: 529
Matches in-memory data: True


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ساخت گزارش کیفیت `Extraction V2`

در این مرحله یک گزارش خلاصه و قابل استناد برای مستندسازی فنی می‌سازیم.

گزارش شامل این موارد خواهد بود:

- تعداد کل رکوردها
- تعداد `source_id` یکتا
- تعداد `article_id` یکتا
- تعداد عنوان، متن، نویسنده و Hash خالی
- حداقل، حداکثر و میانگین طول متن
- تعداد مقالات `primary_author`
- تعداد مقالات `supervised`
- تعداد کل Tagهای استخراج‌شده

</div>

In [77]:
# Code Cell 78 - Build extraction quality summary

body_lengths = [
    len(record["body_text"] or "")
    for record in extracted_website_records
]

total_tags = sum(
    len(record["tags"])
    for record in extracted_website_records
)

extraction_quality_summary = {
    "total_records": len(
        extracted_website_records
    ),
    "unique_source_ids": len({
        record["source_id"]
        for record in extracted_website_records
    }),
    "unique_article_ids": len({
        record["article_id"]
        for record in extracted_website_records
    }),
    "missing_titles": sum(
        not record["title"]
        for record in extracted_website_records
    ),
    "missing_bodies": sum(
        not record["body_text"]
        for record in extracted_website_records
    ),
    "missing_authors": sum(
        not record["author"]
        for record in extracted_website_records
    ),
    "missing_raw_hashes": sum(
        not record["raw_content_hash"]
        for record in extracted_website_records
    ),
    "min_body_length": min(
        body_lengths
    ),
    "max_body_length": max(
        body_lengths
    ),
    "average_body_length": round(
        sum(body_lengths) / len(body_lengths),
        2,
    ),
    "primary_author_records": sum(
        record["content_origin"]
        == "primary_author"
        for record in extracted_website_records
    ),
    "supervised_records": sum(
        record["content_origin"]
        == "supervised"
        for record in extracted_website_records
    ),
    "total_tags": total_tags,
}

for key, value in extraction_quality_summary.items():
    print(f"{key}: {value}")

total_records: 529
unique_source_ids: 529
unique_article_ids: 529
missing_titles: 0
missing_bodies: 0
missing_authors: 0
missing_raw_hashes: 0
min_body_length: 224
max_body_length: 9723
average_body_length: 1758.99
primary_author_records: 76
supervised_records: 453
total_tags: 1558


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ذخیره گزارش کیفیت `Extraction V2`

حالا همین خلاصه را به‌صورت یک فایل مستقل در بخش گزارش‌های Phase 2 ذخیره می‌کنیم.

این فایل بعداً برای مستندسازی فنی، کنترل کیفیت و مقایسه نسخه‌های Dataset قابل استفاده خواهد بود.

</div>

In [78]:
# Code Cell 79 - Save extraction quality report

quality_report_path = (REPORTS_ROOT / "phase2/data_quality/website_extraction_v2.json")

quality_report_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with quality_report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        extraction_quality_summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Saved:",
    quality_report_path,
)

print(
    "Size bytes:",
    quality_report_path.stat().st_size,
)

Saved: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/reports/phase2/data_quality/website_extraction_v2.json
Size bytes: 350


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی فایل گزارش کیفیت

حالا گزارش ذخیره‌شده را دوباره از دیسک می‌خوانیم تا مطمئن شویم فایل سالم است و دقیقاً با Summary داخل حافظه برابر است.

</div>

In [79]:
# Code Cell 80 - Reload and verify quality report

with quality_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    reloaded_quality_report = json.load(file)


print(
    "Matches in-memory summary:",
    reloaded_quality_report
    == extraction_quality_summary,
)

print()
print(reloaded_quality_report)

Matches in-memory summary: True

{'total_records': 529, 'unique_source_ids': 529, 'unique_article_ids': 529, 'missing_titles': 0, 'missing_bodies': 0, 'missing_authors': 0, 'missing_raw_hashes': 0, 'min_body_length': 224, 'max_body_length': 9723, 'average_body_length': 1758.99, 'primary_author_records': 76, 'supervised_records': 453, 'total_tags': 1558}


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ساخت گزارش نهایی Website Ingestion

این گزارش وضعیت فعلی Pipeline وب‌سایت را ثبت می‌کند:

- Discovery
- Raw Crawl
- Raw Validation
- Extraction V2
- Data Quality

این فایل برای مستندسازی و ثبت وضعیت قابل بازتولید Pipeline استفاده می‌شود.

</div>

In [80]:
# Code Cell 81 - Build website ingestion report

website_ingestion_report = {
    "source_type": "website",
    "pipeline_stage": "website_ingestion_v2",
    "status": "completed",
    "total_discovered": 529,
    "total_raw_html": 529,
    "total_raw_valid": 529,
    "total_extracted": 529,
    "manifest_path": (
        "data/phase2/manifests/"
        "website_manifest_v1.json"
    ),
    "raw_root": (
        "data/phase2/raw/website"
    ),
    "extracted_path": (
        "data/phase2/interim/"
        "website_extracted_v2.json"
    ),
    "quality_report_path": (
        "reports/phase2/data_quality/"
        "website_extraction_v2.json"
    ),
    "primary_author_records": (
        extraction_quality_summary[
            "primary_author_records"
        ]
    ),
    "supervised_records": (
        extraction_quality_summary[
            "supervised_records"
        ]
    ),
    "missing_titles": (
        extraction_quality_summary[
            "missing_titles"
        ]
    ),
    "missing_bodies": (
        extraction_quality_summary[
            "missing_bodies"
        ]
    ),
    "missing_authors": (
        extraction_quality_summary[
            "missing_authors"
        ]
    ),
    "missing_raw_hashes": (
        extraction_quality_summary[
            "missing_raw_hashes"
        ]
    ),
}


for key, value in website_ingestion_report.items():
    print(f"{key}: {value}")

source_type: website
pipeline_stage: website_ingestion_v2
status: completed
total_discovered: 529
total_raw_html: 529
total_raw_valid: 529
total_extracted: 529
manifest_path: data/phase2/manifests/website_manifest_v1.json
raw_root: data/phase2/raw/website
extracted_path: data/phase2/interim/website_extracted_v2.json
quality_report_path: reports/phase2/data_quality/website_extraction_v2.json
primary_author_records: 76
supervised_records: 453
missing_titles: 0
missing_bodies: 0
missing_authors: 0
missing_raw_hashes: 0


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## ذخیره گزارش نهایی Website Ingestion

حالا Snapshot نهایی این مرحله را در پوشه گزارش‌های Ingestion ذخیره می‌کنیم.

این فایل مشخص می‌کند که کل مسیر:

`Discovery -> Raw Crawl -> Validation -> Extraction -> Quality Check`

برای هر `529` مقاله با موفقیت کامل شده است.

</div>

In [81]:
# Code Cell 82 - Save website ingestion report

ingestion_report_path = (REPORTS_ROOT / "phase2/ingestion/website_ingestion_v2.json")

ingestion_report_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with ingestion_report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        website_ingestion_report,
        file,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Saved:",
    ingestion_report_path,
)

print(
    "Size bytes:",
    ingestion_report_path.stat().st_size,
)

Saved: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/reports/phase2/ingestion/website_ingestion_v2.json
Size bytes: 623


<div dir="rtl" style="text-align: right; line-height: 1.8;">

## بررسی فایل گزارش نهایی Ingestion

حالا فایل گزارش را دوباره از دیسک می‌خوانیم تا مطمئن شویم Snapshot نهایی سالم و دقیقاً برابر با داده داخل حافظه است.

</div>

In [82]:
# Code Cell 83 - Reload and verify website ingestion report

with ingestion_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    reloaded_ingestion_report = json.load(file)


print(
    "Matches in-memory report:",
    reloaded_ingestion_report
    == website_ingestion_report,
)

print()
print(reloaded_ingestion_report)

Matches in-memory report: True

{'source_type': 'website', 'pipeline_stage': 'website_ingestion_v2', 'status': 'completed', 'total_discovered': 529, 'total_raw_html': 529, 'total_raw_valid': 529, 'total_extracted': 529, 'manifest_path': 'data/phase2/manifests/website_manifest_v1.json', 'raw_root': 'data/phase2/raw/website', 'extracted_path': 'data/phase2/interim/website_extracted_v2.json', 'quality_report_path': 'reports/phase2/data_quality/website_extraction_v2.json', 'primary_author_records': 76, 'supervised_records': 453, 'missing_titles': 0, 'missing_bodies': 0, 'missing_authors': 0, 'missing_raw_hashes': 0}
